# Notebook 01 — Exploration and Data Preparation

## Step A — Data Preparation

I’ll prepare the datasets for my Birmingham-centred study one source at a time. I’ll explain my geography and field choices before applying them, and keep the original source files unchanged.

First, I’m checking the tools before opening the toolbox. The code below checks my Python version, whether I’m running inside a virtual environment, and whether the initial libraries are available. It reads environment information; it does not load or change datasets. `version()` reports each installed package’s version in a small table.

In [1]:
import sys
from importlib.metadata import version
import pandas as pd

print("Python version:", sys.version.split()[0])
print("Running inside a virtual environment:", sys.prefix != sys.base_prefix)
print("Environment folder name:", __import__("pathlib").Path(sys.prefix).name)

packages = ["pandas", "openpyxl", "geopandas", "matplotlib", "ipykernel"]
pd.DataFrame({
    "Library": packages,
    "Installed version": [version(package) for package in packages],
})

Python version: 3.14.7
Running inside a virtual environment: True
Environment folder name: .venv


,Library,Installed version
0,pandas,3.0.6
1,openpyxl,3.1.5
2,geopandas,1.2.0
3,matplotlib,3.11.2
4,ipykernel,7.4.0


### How I read the result

I expect Python **3.14.7**, `True` for the virtual-environment check, and `.venv` as the environment folder name. The table lists installed library versions, not dataset results.

If those checks differ, I’ll check my selected notebook kernel before continuing. A folder called `.venv` is a useful clue, but the name alone does not prove it belongs to this project.

At this point I haven’t filtered data, selected columns or exported files.

## 1. Healthy Life Expectancy (HLE) — understand the source

HLE means expected years spent in good health. It is **my entry point into the data**: I’ll lay this dataset over the city map, look for variation between areas, then ask which other area characteristics might help explain the patterns.  I chose HLE instead of simply Life Expectancy (LE) because HLE provides a more nuanced view of the health of the population, and it can show greater variation in health outcomes across different areas. So HLE as my master dataset comes first; qualifications and other contextual datasets follow it.

The ONS workbook contains several sheets. **Sheet `2` contains healthy life expectancy at birth for 2019–2023**, separately for males and females. Sheet `1` contains life expectancy; sheet `3` contains disability-free life expectancy. Similar names, different measures—so I need the right sheet.

`pd.read_excel()` reads a worksheet into a DataFrame, a table held in Python. `header=6` uses the seventh worksheet row as the column headings, skipping the titles and notes above it. I locate my project folder, read the HLE sheet, then display its fields and a short preview. The source stays unchanged.

In [2]:
from pathlib import Path
from IPython.display import display

# Find the project from the notebook's current folder.
working_folder = Path.cwd()
project_folder = next(
    folder for folder in [working_folder, *working_folder.parents]
    if (folder / "Data/Raw").is_dir() or (folder / "data/raw").is_dir()
)
data_folder = project_folder / ("Data" if (project_folder / "Data").is_dir() else "data")

def project_data(relative_path):
    """Locate a data file or folder in either project layout."""
    parts = Path(relative_path).parts
    if data_folder.name == "data":
        parts = (parts[0].lower(), *parts[1:])
    return data_folder.joinpath(*parts)

def raw_file(filename):
    """Find an untouched source in the project's raw collection."""
    candidates = sorted(project_data("Raw").rglob(filename))
    if not candidates:
        # The working collection also contains the original Unit 2 downloads.
        candidates = sorted((project_folder.parent.parent / "Datasets/Raw").rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"Raw source not found: {filename}")
    return candidates[0]

hle_workbook = raw_file("ons_hsle_msoa_2019_2023.xlsx")
# Sheet 2 is HLE; its column headings start on worksheet row 7.
hle_raw = pd.read_excel(hle_workbook, sheet_name="2", header=6)

print(f"Source: {len(hle_raw):,} rows; {len(hle_raw.columns)} fields")
display(pd.DataFrame({"Source field": hle_raw.columns}))
display(hle_raw[["Area code", "Area name", "Sex", "HLE"]].head())

Source: 15,355 rows; 13 fields


,Source field
0,Period
1,Country
2,Area type
3,Area code
4,Area name
5,Sex
6,Sex code
7,HLE
8,LCI
9,UCI


,Area code,Area name,Sex,HLE
0,E02000001,City of London 001,Male,72.4
1,E02000001,City of London 001,Female,74.6
2,E02000002,Barking and Dagenham 001,Male,60.2
3,E02000002,Barking and Dagenham 001,Female,60.1
4,E02000003,Barking and Dagenham 002,Male,65.0


### What I need from this HLE dataset

One observation is **one MSOA, one sex and one period**. I keep the area code as the stable link to the map and other datasets, and keep male and female estimates separate.  I keep the compact view focused and retain the other fields separately.

My compact table needs four fields: **area code, local area name, sex and HLE in years**. All HLE estimates cover **2019–2023**, so I state that here instead of repeating it in a column. I’ll apply familiar local names after inspecting the source labels below.

I keep confidence intervals, proportion of life spent in good health and both imputation flags in a supporting table. The intervals describe uncertainty; the flags record source estimation information. I don’t automatically discard flagged estimates or turn missing values into zeros.

Country, area type and sex code also stay in the supporting table. Useful evidence can stay behind the scenes without cluttering the compact view.

First, I select the **660 MSOAs covered by my existing boundary file**. That is available coverage, not my final radius or study boundary. `isin()` checks each area code against that coverage; `.copy()` creates a separate selected table.

In [3]:
import geopandas as gpd

boundary_file = raw_file("ons_msoa21_boundaries_exploratory_west_midlands.geojson")
area_boundaries = gpd.read_file(boundary_file)
coverage_codes = set(area_boundaries["MSOA21CD"])
assert area_boundaries["MSOA21CD"].is_unique, "Repeated boundary codes"

# Match statistical area codes, keeping both sexes within the available map coverage.
hle_coverage = hle_raw.loc[
    hle_raw["Area code"].isin(coverage_codes)
    & hle_raw["Area type"].eq("MSOA")
    & hle_raw["Period"].eq("2019 to 2023")
].copy()
missing_hle_codes = coverage_codes - set(hle_coverage["Area code"])
assert not missing_hle_codes, f"Coverage areas without HLE rows: {missing_hle_codes}"
# One row per area and sex is expected for this period.
assert not hle_coverage.duplicated(["Area code", "Sex", "Period"]).any()
assert hle_coverage["Country"].eq("England").all()
assert set(hle_coverage["Sex"]) == {"Male", "Female"}
assert hle_coverage.groupby("Area code")["Sex"].nunique().eq(2).all()

print(f"Available areas: {hle_coverage['Area code'].nunique():,}")
print(f"HLE observations: {len(hle_coverage):,} — male and female rows kept separately")
print(f"Coverage areas without HLE rows: {len(missing_hle_codes)}")

Available areas: 660
HLE observations: 1,320 — male and female rows kept separately
Coverage areas without HLE rows: 0


### Boil HLE down to the table I need

I keep all source fields for the selected coverage in `hle_support`. I then take four fields for `hle_display` and give their columns shorter names. **The area names at this stage are still the original ONS labels.** I’ll deal with those immediately after the preview.

`hle_years` remains the published value: I haven’t recalculated, rounded or filled it. The checks report missing estimates and source flags. I’m preparing tables in memory here, without exporting files or joining contextual data.

In [4]:
# Keep the source evidence separately before narrowing the displayed table.
hle_support = hle_coverage.copy()
hle_display = hle_support[
    ["Area code", "Area name", "Sex", "HLE"]
].rename(columns={
    "Area code": "area_code", "Area name": "area_name",
    "Sex": "sex", "HLE": "hle_years",
}).copy()

assert len(hle_display) == len(hle_support)
assert not hle_display.duplicated(["area_code", "sex"]).any()
assert pd.api.types.is_numeric_dtype(hle_display["hle_years"])
print(f"Fields: {len(hle_support.columns)} supporting → {len(hle_display.columns)} compact")
print(f"Missing HLE estimates: {hle_display['hle_years'].isna().sum()}")
display(hle_display.head(10))
display(hle_support[["LE imputed", "Health prevalence imputed"]].value_counts(dropna=False).rename("Rows").reset_index())

Fields: 13 supporting → 4 compact
Missing HLE estimates: 0


,area_code,area_name,sex,hle_years
3492,E02001827,Birmingham 001,Male,68.4
3493,E02001827,Birmingham 001,Female,68.3
3494,E02001828,Birmingham 002,Male,69.5
3495,E02001828,Birmingham 002,Female,72.6
3496,E02001829,Birmingham 003,Male,73.3
3497,E02001829,Birmingham 003,Female,73.3
3498,E02001830,Birmingham 004,Male,71.7
3499,E02001830,Birmingham 004,Female,72.4
3500,E02001831,Birmingham 005,Male,63.7
3501,E02001831,Birmingham 005,Female,63.9


,LE imputed,Health prevalence imputed,Rows
0,0,0,1311
1,1,0,8
2,2,0,1


### Area names — why I’m changing them and how

Birmingham 001, Birmingham 002, Birmingham 003: fine for a statistical filing cabinet, less helpful for recognising a place. **I’m replacing numbered ONS labels with familiar local names**, such as Sparkbrook, Tyseley and Greet.

My target audience is people, both professional and general public, who should all know the city. I want them to recognise the places on the map and concentrate on the visualised data differences, rather than trying to decode a numbering system written for a machine.

My plan is to:

1. Use each **MSOA area code** to identify its map boundary.
2. Use a familiar local name for that area, with a qualifier such as East or West where helpful. I’ll review the labels against the boundaries because an MSOA can include several neighbourhoods.
3. Match the naming lookup to my HLE table **by area code** and replace the displayed area name.
4. Use the familiar names in my compact table, map labels and comparisons.

I already have a starting lookup from my Concept B prototype visualisation, so I don’t need to invent the names again. The labels are provisional descriptions of the statistical areas. Changing a label does not change its boundary, area code or HLE value. I’ll flag any missing or unfamiliar names for review.

### Apply the familiar local names

I’m using **Concept B’s naming lookup**, copied into `data/reference/Area Names/`. It covers all 660 MSOA codes in my current geographical coverage.

`json.load()` reads that saved lookup. Each code has a `display_name` and, where needed, a `qualifier` such as East or West. I combine those parts into a readable name. `map()` matches each `area_code` to that name, then I replace the displayed name. Male and female rows for the same area receive the same label.

I check that every code has a non-empty name before applying the change. The ten rows below show **local_area_name**, area code, sex and HLE. **All estimates cover 2019–2023.** The numbered ONS names have done their job; I don’t need them alongside the new labels.

The starting names come from the House of Commons Library MSOA Names lookup, adapted for Concept B. `naming-source.json` and `msoa-names-source.csv` retain the attribution, licence and published starting names.

In [5]:
import json

local_names_file = project_data("Reference/Area Names/locality-names.json")
with local_names_file.open(encoding="utf-8") as names_file:
    local_names = json.load(names_file)

# Keep useful qualifiers: Acocks Green East and West must remain distinguishable.
local_name_by_code = {
    area_code: " ".join(
        part.strip() for part in [entry["display_name"], entry.get("qualifier", "")]
        if part.strip()
    )
    for area_code, entry in local_names.items()
}
missing_name_codes = set(hle_display["area_code"]) - set(local_name_by_code)
assert not missing_name_codes, f"Area codes without local names: {sorted(missing_name_codes)}"

# Match by code so the familiar name stays attached to the correct HLE estimate.
new_area_names = hle_display["area_code"].map(local_name_by_code)
assert new_area_names.notna().all() and new_area_names.str.strip().ne("").all()
# Check that replacing names changes no other values.
hle_before_renaming = hle_display.drop(columns="area_name").copy()
hle_display["area_name"] = new_area_names
pd.testing.assert_frame_equal(hle_display.drop(columns="area_name"), hle_before_renaming)

display(hle_display.head(10).rename(columns={"area_name": "local_area_name"}))
print(f"Local names applied to {hle_display['area_code'].nunique():,} areas; {len(hle_display):,} HLE rows.")
print("Area codes and HLE values unchanged.")

,area_code,local_area_name,sex,hle_years
3492,E02001827,Hill Hook,Male,68.4
3493,E02001827,Hill Hook,Female,68.3
3494,E02001828,Four Oaks,Male,69.5
3495,E02001828,Four Oaks,Female,72.6
3496,E02001829,Little Sutton,Male,73.3
3497,E02001829,Little Sutton,Female,73.3
3498,E02001830,Sutton Coldfield North,Male,71.7
3499,E02001830,Sutton Coldfield North,Female,72.4
3500,E02001831,Reddicap Heath,Male,63.7
3501,E02001831,Reddicap Heath,Female,63.9


Local names applied to 660 areas; 1,320 HLE rows.
Area codes and HLE values unchanged.


### My HLE preparation so far

I’ve loaded the **2019–2023** HLE sheet, selected the available local coverage, reduced it to a compact table and **applied Concept B’s familiar local names**. The ten-row preview above shows the table I now have.

`hle_display` contains the local names I want people to read. My supporting table retains the source evidence, uncertainty and flags. Male and female estimates remain separate.

I still need to review the provisional labels and decide the final geographical selection. The 660-area coverage is a starting envelope, not a settled study boundary. I haven’t yet tested associations or fitted a model.

### Save my cleaned HLE dataset

I now have the compact and clean HLE table I need, so it’s time to give it a home outside notebook memory. I’ll save it as **`data/cleaned/HLE/hle_2019_2023_msoa.csv`**. `Raw` holds the untouched source; `Cleaned` will hold all my prepared datasets, this one boiled down to the essentials and with familiar local names and the four columns I want to use.

**All HLE estimates cover 2019–2023.** The file contains `area_code`, `local_area_name`, `sex` and `hle_years`. “Cleaned” means I have selected the coverage and fields and changed the display names; I haven’t altered the published HLE values.

`mkdir()` creates the destination folder if needed. `to_csv()` saves the table as a CSV file, which can be opened by pandas and other data tools. `index=False` leaves out the notebook’s row numbers—they aren’t another data field.

If I run this again, I keep a timestamped copy of the existing CSV before replacing it. Finally, I read the saved file back and check its columns, row count and values against the table I intended to save.

In [6]:
from datetime import datetime
import shutil

clean_hle = hle_display.rename(columns={"area_name": "local_area_name"}).copy()
clean_hle = clean_hle[["area_code", "local_area_name", "sex", "hle_years"]]
assert not clean_hle.duplicated(["area_code", "sex"]).any()
assert clean_hle["local_area_name"].notna().all()

clean_folder = project_data("Cleaned/HLE")
clean_folder.mkdir(parents=True, exist_ok=True)
clean_file = clean_folder / "hle_2019_2023_msoa.csv"

# Retain the previous saved dataset before replacing it on a rerun.
if clean_file.exists():
    backup_folder = project_folder / "VS_CODE/Backups/Outputs/HLE"
    backup_folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    shutil.copy2(clean_file, backup_folder / f"hle_2019_2023_msoa - {stamp}.csv")

clean_hle.to_csv(clean_file, index=False)
# Read the file back to check what actually reached disk.
saved_hle = pd.read_csv(clean_file)
pd.testing.assert_frame_equal(saved_hle, clean_hle.reset_index(drop=True))
print("Saved: Data/Cleaned/HLE/hle_2019_2023_msoa.csv")
print(f"Verified: {len(saved_hle):,} rows, {saved_hle['area_code'].nunique():,} areas, {len(saved_hle.columns)} columns.")
display(saved_hle.head(10))

Saved: Data/Cleaned/HLE/hle_2019_2023_msoa.csv
Verified: 1,320 rows, 660 areas, 4 columns.


,area_code,local_area_name,sex,hle_years
0,E02001827,Hill Hook,Male,68.4
1,E02001827,Hill Hook,Female,68.3
2,E02001828,Four Oaks,Male,69.5
3,E02001828,Four Oaks,Female,72.6
4,E02001829,Little Sutton,Male,73.3
5,E02001829,Little Sutton,Female,73.3
6,E02001830,Sutton Coldfield North,Male,71.7
7,E02001830,Sutton Coldfield North,Female,72.4
8,E02001831,Reddicap Heath,Male,63.7
9,E02001831,Reddicap Heath,Female,63.9


## 2. Household deprivation — understand the source

I’m preparing the data I already have before hunting for more. First up is **Census 2021 TS011: household deprivation**. This measures disadvantage in four dimensions: education, employment, health/disability and housing. It does **not** measure household debt or simply income poverty.

It’s useful because I can compare the share of households facing several kinds of disadvantage with the HLE pattern on my map. That gives me a question to investigate, not an explanation delivered on a plate. The health/disability dimension overlaps HLE’s subject, so an association may partly reflect that overlap.

I read the MSOA CSV directly from the source ZIP. The source contains household **counts**; I need percentages to compare areas with different numbers of households.

In [7]:
from zipfile import ZipFile

# This collected source is in the Unit 2 raw collection, above the project folder.
deprivation_zip = raw_file("census2021-ts011.zip")
with ZipFile(deprivation_zip) as archive:
    with archive.open("census2021-ts011-msoa.csv") as source_file:
        deprivation_raw = pd.read_csv(source_file)
print(f"Source: {len(deprivation_raw):,} MSOA rows; {len(deprivation_raw.columns)} fields")
display(pd.DataFrame({"Source field": deprivation_raw.columns}))
display(deprivation_raw.loc[deprivation_raw["geography"].str.startswith("Birmingham", na=False)].iloc[:5, :4])

Source: 7,264 MSOA rows; 9 fields


,Source field
0,date
1,geography
2,geography code
3,Household deprivation: Total: All households; ...
4,Household deprivation: Household is not depriv...
5,Household deprivation: Household is deprived i...
6,Household deprivation: Household is deprived i...
7,Household deprivation: Household is deprived i...
8,Household deprivation: Household is deprived i...


,date,geography,geography code,Household deprivation: Total: All households; measures: Value
2922,2021,Birmingham 001,E02001827,2634
2923,2021,Birmingham 002,E02001828,3130
2924,2021,Birmingham 003,E02001829,3569
2925,2021,Birmingham 004,E02001830,3988
2926,2021,Birmingham 005,E02001831,4493


### Select my geographical coverage first

I first select the same **660 MSOA codes** as my HLE table. This filters the places, **not the columns**: the selected source table still contains all **nine fields**.

Before choosing any compact measures, I want to see those fields and their values. The next displays show the full field list and a five-area Birmingham sample. These are the first Birmingham rows in source order, not the most deprived areas. The full dataset still contains all 660 areas. I turn the sample sideways so the long Census headings run down the page instead of disappearing off the right edge.

In [8]:
assert deprivation_raw["geography code"].is_unique, "Repeated source area codes"
assert deprivation_raw["date"].astype(str).eq("2021").all()
deprivation_support = deprivation_raw.loc[
    deprivation_raw["geography code"].isin(coverage_codes)
].copy()
assert set(deprivation_support["geography code"]) == coverage_codes, "Missing coverage areas"

print(f"Geographically selected source: {len(deprivation_support):,} areas, {len(deprivation_support.columns)} fields")
field_meanings = [
    "Census year — 2021 for every row",
    "Original statistical area name",
    "MSOA code — my link to HLE and the map",
    "Total households — denominator for percentages",
    "Households deprived in none of the four dimensions",
    "Households deprived in exactly one dimension",
    "Households deprived in exactly two dimensions",
    "Households deprived in exactly three dimensions",
    "Households deprived in all four dimensions",
]
assert len(field_meanings) == len(deprivation_support.columns)
display(pd.DataFrame({
    "Source field": deprivation_support.columns,
    "What it contains": field_meanings,
}))

# Preview Birmingham areas in source order; this is not a deprivation ranking.
# Transpose only the preview so every source field remains visible.
birmingham_deprivation = deprivation_support.loc[
    deprivation_support["geography"].str.startswith("Birmingham", na=False)
]
source_preview = birmingham_deprivation.head(5).set_index("geography code").T
source_preview.index.name = "Source field"
display(source_preview)


Geographically selected source: 660 areas, 9 fields


,Source field,What it contains
0,date,Census year — 2021 for every row
1,geography,Original statistical area name
2,geography code,MSOA code — my link to HLE and the map
3,Household deprivation: Total: All households; ...,Total households — denominator for percentages
4,Household deprivation: Household is not depriv...,Households deprived in none of the four dimens...
5,Household deprivation: Household is deprived i...,Households deprived in exactly one dimension
6,Household deprivation: Household is deprived i...,Households deprived in exactly two dimensions
7,Household deprivation: Household is deprived i...,Households deprived in exactly three dimensions
8,Household deprivation: Household is deprived i...,Households deprived in all four dimensions


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Household deprivation: Total: All households; measures: Value,2634,3130,3569,3988,4493
Household deprivation: Household is not deprived in any dimension; measures: Value,1433,1664,2244,2468,1994
Household deprivation: Household is deprived in one dimension; measures: Value,818,1031,1035,1139,1484
Household deprivation: Household is deprived in two dimensions; measures: Value,327,347,243,321,767
Household deprivation: Household is deprived in three dimensions; measures: Value,55,83,44,59,229
Household deprivation: Household is deprived in four dimensions; measures: Value,1,5,3,1,19


### What I can keep, replace or summarise

I can now see all nine source fields for my selected geography:

- **Date:** 2021 throughout, so I can state it in the text instead of repeating it in the compact table.
- **Geography:** the original numbered name; I can replace it with my familiar local label.
- **Geography code:** I keep this to match areas reliably.
- **Total households:** I need it to calculate percentages, even if it stays in supporting data rather than on screen.
- **Five deprivation categories:** household counts for zero, one, two, three or four dimensions. These categories describe how many dimensions apply, not which ones.

The **nine-field table above is the geographically filtered source**. The **three-field table below is my selected summary**, containing the code, familiar name and one calculated percentage. They are two different tables, not conflicting counts.

I choose the share of households deprived in two or more dimensions to describe several kinds of disadvantage occurring together. I keep the individual category counts so I can revisit this choice later.

### Keep households deprived in two or more dimensions (%)

For this measure, I add the counts for two, three and four dimensions, divide by the source’s total households, then multiply by 100. Percentages make comparisons between differently sized areas more useful than raw counts alone.

I attach the familiar names by MSOA code and keep the source counts separately. This calculation does not tell me which dimensions apply, and the health/disability overlap with HLE still matters.

In [9]:
# Use this source's household denominator, not a population total from another table.
total_field = "Household deprivation: Total: All households; measures: Value"
deprived_fields = [
    f"Household deprivation: Household is deprived in {dimension} dimensions; measures: Value"
    for dimension in ["two", "three", "four"]
]
counts = deprivation_support[[total_field, *deprived_fields]]
assert counts.notna().all().all(), "Missing household counts need review"
assert counts.ge(0).all().all(), "Negative household counts"
assert counts[total_field].gt(0).all(), "Invalid household denominator"
deprived_2plus = counts[deprived_fields].sum(axis=1)
assert deprived_2plus.le(counts[total_field]).all()

deprivation_display = pd.DataFrame({
    "area_code": deprivation_support["geography code"],
    "local_area_name": deprivation_support["geography code"].map(local_name_by_code),
    "households_deprived_2plus_pct": 100 * deprived_2plus / counts[total_field],
})
assert deprivation_display["local_area_name"].notna().all()
assert deprivation_display["area_code"].is_unique
assert deprivation_display["households_deprived_2plus_pct"].between(0, 100).all()
print(f"Prepared: {len(deprivation_display):,} areas; {len(deprivation_display.columns)} compact fields")
display(deprivation_display.loc[
    deprivation_display["area_code"].isin(birmingham_deprivation["geography code"])
].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Prepared: 660 areas; 3 compact fields


,area_code,local_area_name,households_deprived_2plus_pct
2922,E02001827,Hill Hook,14.54
2923,E02001828,Four Oaks,13.90
2924,E02001829,Little Sutton,8.13
2925,E02001830,Sutton Coldfield North,9.55
2926,E02001831,Reddicap Heath,22.59
2927,E02001832,Rectory Park,12.18
2928,E02001833,Kingstanding North,13.36
2929,E02001834,Old Oscott,25.66
2930,E02001835,Sutton Coldfield South,12.18
2931,E02001836,New Oscott,13.21


Preview rounded only; saved percentages retain precision.


### Save my household deprivation dataset

I save the compact table in **`data/cleaned/Household Deprivation/`**, alongside a supporting file containing the source counts. The separate support file keeps the evidence available without turning my compact table into a wall of columns.

I back up either file before replacing it on a rerun, then read each saved CSV back to check it. I haven’t joined this to HLE yet: this is one Census observation per area, while HLE has separate male and female rows.

In [10]:
deprivation_folder = project_data("Cleaned/Household Deprivation")
deprivation_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("household_deprivation_2021_msoa.csv", deprivation_display),
    ("household_deprivation_2021_msoa_support.csv", deprivation_support),
]:
    destination = deprivation_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Household Deprivation"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows, {len(reloaded.columns)} fields")

Saved and verified household_deprivation_2021_msoa.csv: 660 rows, 3 fields
Saved and verified household_deprivation_2021_msoa_support.csv: 660 rows, 9 fields


### My household deprivation preparation so far

I now have a three-column dataset covering the same **660 areas** as HLE, with familiar local names and the percentage of households deprived in two or more dimensions. The counts remain available in a separate supporting file, and the raw source is unchanged.

This is a candidate contextual measure, not a demonstrated explanation of HLE. **Overcrowding is next**, followed by fuel poverty, pollution, economic activity and qualifications. I’ll consider age after those rather than let the easiest download dictate the order.

## 3. Overcrowding — apply my preparation process

My process is **select the relevant geography → inspect every remaining field → select useful measures → check and save**. I keep the source evidence separately. Thinning the table should sharpen the question, not throw away the evidence needed to answer it.

Here I use **Census 2021 TS052: occupancy rating for bedrooms**. It compares the number of bedrooms available with the number required under the Census bedroom standard. It is not simply people divided by bedrooms.

Overcrowding is worth investigating alongside HLE because housing conditions may be associated with health. I haven’t demonstrated that relationship in these areas. This measure describes households with a bedroom shortfall, not every aspect of housing quality.

First I read the MSOA source from its ZIP and select the same 660 area codes as HLE. I keep all columns at this stage.

In [11]:
overcrowding_zip = raw_file("census2021-ts052.zip")
with ZipFile(overcrowding_zip) as archive:
    with archive.open("census2021-ts052-msoa.csv") as source_file:
        overcrowding_raw = pd.read_csv(source_file)
assert overcrowding_raw["geography code"].is_unique
assert overcrowding_raw["date"].astype(str).eq("2021").all()

# Filter geography first; leave all source columns available for inspection.
overcrowding_support = overcrowding_raw.loc[
    overcrowding_raw["geography code"].isin(coverage_codes)
].copy()
assert set(overcrowding_support["geography code"]) == coverage_codes
print(f"National source: {len(overcrowding_raw):,} areas")
print(f"Selected coverage: {len(overcrowding_support):,} areas; {len(overcrowding_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 9 fields


### Inspect the fields before thinning the columns

The field list explains all nine source columns. The sideways preview shows five **Birmingham** areas in source order, not a ranking of overcrowding. I keep all 660 areas in the dataset.

The positive ratings mean spare bedrooms; zero means the number available matches the standard. Negative ratings mean a shortfall. Zero is a real category, not missing data.

In [12]:
overcrowding_meanings = [
    "Census year — 2021 throughout",
    "Original statistical area name",
    "MSOA code — link to HLE and the map",
    "Total households — denominator for percentages",
    "Households with at least two bedrooms above the standard",
    "Households with one bedroom above the standard",
    "Households with exactly the number of bedrooms required",
    "Households with one fewer bedroom than required",
    "Households with at least two fewer bedrooms than required",
]
assert len(overcrowding_meanings) == len(overcrowding_support.columns)
display(pd.DataFrame({"Source field": overcrowding_support.columns,
                      "What it contains": overcrowding_meanings}))
# Transpose the preview so the full set of fields fits vertically.
birmingham_overcrowding = overcrowding_support.loc[
    overcrowding_support["geography"].str.startswith("Birmingham", na=False)
]
overcrowding_preview = birmingham_overcrowding.head(5).set_index("geography code").T
overcrowding_preview.index.name = "Source field"
display(overcrowding_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Occupancy rating for bedrooms: Total: All hous...,Total households — denominator for percentages
4,Occupancy rating for bedrooms: Occupancy ratin...,Households with at least two bedrooms above th...
5,Occupancy rating for bedrooms: Occupancy ratin...,Households with one bedroom above the standard
6,Occupancy rating for bedrooms: Occupancy ratin...,Households with exactly the number of bedrooms...
7,Occupancy rating for bedrooms: Occupancy ratin...,Households with one fewer bedroom than required
8,Occupancy rating for bedrooms: Occupancy ratin...,Households with at least two fewer bedrooms th...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Occupancy rating for bedrooms: Total: All households,2633,3130,3571,3987,4493
Occupancy rating for bedrooms: Occupancy rating of bedrooms: +2 or more,1229,1307,2000,2351,1482
Occupancy rating for bedrooms: Occupancy rating of bedrooms: +1,865,1166,1109,1027,1474
Occupancy rating for bedrooms: Occupancy rating of bedrooms: 0,490,605,417,546,1329
Occupancy rating for bedrooms: Occupancy rating of bedrooms: -1,43,46,39,55,184
Occupancy rating for bedrooms: Occupancy rating of bedrooms: -2 or less,6,6,6,8,24


### Keep a punchy overcrowding measure

For my compact table, I use **households with a bedroom shortfall (%)**. I add the `-1` and `-2 or less` household counts, divide by the source’s total households and multiply by 100. This is my overcrowding measure under the bedroom standard.

I keep **area code, local area name and overcrowding percentage**. The Census year is **2021**, stated here rather than repeated down a column. The original names become familiar local labels. The total and category counts remain in supporting data; spare-bedroom categories don’t need to clutter this particular compact view.

The next preview shows ten Birmingham areas. Values are rounded for reading only.

**My decision:** overcrowding is the housing hypothesis I want to investigate now, so I’m leaving spare-bedroom measures out of the compact dataset. If bedroom occupancy shows a rich association with HLE, I may return to the spare-bedroom categories to explore the wider pattern. Their original counts remain in the supporting file; I am not adding them as features at this stage.

In [13]:
bedroom_total = "Occupancy rating for bedrooms: Total: All households"
shortfall_fields = [
    "Occupancy rating for bedrooms: Occupancy rating of bedrooms: -1",
    "Occupancy rating for bedrooms: Occupancy rating of bedrooms: -2 or less",
]
# Use the matching household denominator, not a population count.
bedroom_counts = overcrowding_support[[bedroom_total, *shortfall_fields]]
assert bedroom_counts.notna().all().all(), "Missing counts need review"
assert bedroom_counts.ge(0).all().all(), "Negative household counts"
assert bedroom_counts[bedroom_total].gt(0).all(), "Invalid household denominator"
shortfall_count = bedroom_counts[shortfall_fields].sum(axis=1)
assert shortfall_count.le(bedroom_counts[bedroom_total]).all()

overcrowding_display = pd.DataFrame({
    "area_code": overcrowding_support["geography code"],
    "local_area_name": overcrowding_support["geography code"].map(local_name_by_code),
    "overcrowded_households_pct": 100 * shortfall_count / bedroom_counts[bedroom_total],
})
assert overcrowding_display["area_code"].is_unique
assert overcrowding_display["local_area_name"].notna().all()
assert overcrowding_display["overcrowded_households_pct"].between(0, 100).all()
print(f"Compact table: {len(overcrowding_display):,} areas; {len(overcrowding_display.columns)} fields")
display(overcrowding_display.loc[
    overcrowding_display["area_code"].isin(birmingham_overcrowding["geography code"])
].head(10).round(2))

Compact table: 660 areas; 3 fields


,area_code,local_area_name,overcrowded_households_pct
1746,E02001827,Hill Hook,1.86
1747,E02001828,Four Oaks,1.66
1748,E02001829,Little Sutton,1.26
1749,E02001830,Sutton Coldfield North,1.58
1750,E02001831,Reddicap Heath,4.63
1751,E02001832,Rectory Park,1.89
1752,E02001833,Kingstanding North,2.49
1753,E02001834,Old Oscott,5.73
1754,E02001835,Sutton Coldfield South,1.81
1755,E02001836,New Oscott,1.67


### Check and save my overcrowding dataset

I save the three-column table in **`data/cleaned/Overcrowding/`**, with the nine-field source-count table in a separate supporting CSV. On reruns I back up existing files before replacing them, then read the saved files back to verify the contents.

My raw source stays unchanged. I have prepared an area-level measure, not tested a relationship with HLE.

In [14]:
overcrowding_folder = project_data("Cleaned/Overcrowding")
overcrowding_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("overcrowding_2021_msoa.csv", overcrowding_display),
    ("overcrowding_2021_msoa_support.csv", overcrowding_support),
]:
    destination = overcrowding_folder / filename
    # Preserve the previous export before replacing it.
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Overcrowding"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified overcrowding_2021_msoa.csv: 660 rows; 3 fields
Saved and verified overcrowding_2021_msoa_support.csv: 660 rows; 9 fields


## 4. Fuel poverty — understand the source

Next I apply the same process: **select geography → inspect fields → choose the useful measure → check and save**.

This workbook contains **2024 fuel-poverty estimates**, published in 2026. I use **Table 4**, which reports households at **LSOA** level. LSOAs are smaller than the MSOAs in my HLE dataset, so I need to combine them into their parent MSOAs.

Fuel poverty is useful to investigate because difficulty keeping a home adequately heated may be associated with health. These are the source’s fuel-poverty estimates, not a direct measure of every household’s unpaid bills. Their 2024 period comes after the 2019–2023 HLE period, which limits interpretation of any relationship.

The workbook cautions that small-area estimates should describe general patterns, rather than tiny differences between similar areas or trends within an LSOA. Aggregating them does not remove that uncertainty.

In [15]:
fuel_workbook = raw_file("fuel_poverty_2024_tables_published_2026.xlsx")
# Table 4 headings are on row 3; exclude titles and footer rows using valid area codes.
fuel_sheet = pd.read_excel(fuel_workbook, sheet_name="Table 4", header=2)
fuel_raw = fuel_sheet.loc[
    fuel_sheet["LSOA Code"].astype("string").str.fullmatch(r"E010\d{5}", na=False)
].copy()
assert fuel_raw["LSOA Code"].is_unique

lookup_file = project_data("Prepared/Geography/ons_lsoa21_to_msoa21_lad22_exploratory_west_midlands.csv")
fuel_lookup = pd.read_csv(lookup_file)
fuel_lookup = fuel_lookup.loc[fuel_lookup["MSOA21CD"].isin(coverage_codes)].copy()
assert fuel_lookup["LSOA21CD"].is_unique
assert set(fuel_lookup["MSOA21CD"]) == coverage_codes
# Require every expected smaller area: never silently calculate a partial MSOA.
missing_lsoas = set(fuel_lookup["LSOA21CD"]) - set(fuel_raw["LSOA Code"])
assert not missing_lsoas, f"Expected LSOAs missing from fuel poverty source: {sorted(missing_lsoas)}"
fuel_selected = fuel_raw.loc[fuel_raw["LSOA Code"].isin(fuel_lookup["LSOA21CD"])].copy()
print(f"National source: {len(fuel_raw):,} LSOAs")
print(f"Selected source: {len(fuel_selected):,} LSOAs; {len(fuel_selected.columns)} original fields")
print(f"Parent MSOAs: {fuel_lookup['MSOA21CD'].nunique():,}; missing expected LSOAs: {len(missing_lsoas)}")

National source: 33,755 LSOAs
Selected source: 3,205 LSOAs; 8 original fields
Parent MSOAs: 660; missing expected LSOAs: 0


### Inspect all the selected source fields

I show the eight original fields before choosing the compact measure. The sideways sample uses five Birmingham LSOAs in source order, not the highest fuel-poverty estimates.

The source contains names and wider-area labels for orientation, plus the counts and published percentage. My lookup adds the MSOA code needed for aggregation; that is supporting geography, not another feature to analyse.

In [16]:
fuel_meanings = [
    "LSOA code — match to the MSOA lookup",
    "Original LSOA name",
    "Source local-authority code",
    "Source local-authority name",
    "Source region label",
    "Estimated total households — denominator",
    "Estimated households in fuel poverty — numerator",
    "Published LSOA fuel-poverty percentage",
]
assert len(fuel_meanings) == len(fuel_selected.columns)
display(pd.DataFrame({"Source field": fuel_selected.columns, "What it contains": fuel_meanings}))
fuel_birmingham_preview = fuel_selected.loc[fuel_selected["Local Authority Name"].eq("Birmingham")].head(5).set_index("LSOA Code").T
fuel_birmingham_preview.index.name = "Source field"
display(fuel_birmingham_preview)

,Source field,What it contains
0,LSOA Code,LSOA code — match to the MSOA lookup
1,LSOA Name,Original LSOA name
2,Local Authority Code,Source local-authority code
3,Local Authority Name,Source local-authority name
4,Region,Source region label
5,Number of households,Estimated total households — denominator
6,Number of households in fuel poverty,Estimated households in fuel poverty — numerator
7,Proportion of households fuel poor (%),Published LSOA fuel-poverty percentage


LSOA Code,E01008881,E01008882,E01008883,E01008884,E01008885
Source field,,,,,
LSOA Name,Birmingham 067A,Birmingham 066A,Birmingham 078A,Birmingham 078B,Birmingham 076A
Local Authority Code,E08000025,E08000025,E08000025,E08000025,E08000025
Local Authority Name,Birmingham,Birmingham,Birmingham,Birmingham,Birmingham
Region,West Midlands,West Midlands,West Midlands,West Midlands,West Midlands
Number of households,612.0,696.0,664.0,740.0,525.0
Number of households in fuel poverty,88.0,114.0,112.0,175.0,54.0
Proportion of households fuel poor (%),14.379085,16.37931,16.86747,23.648649,10.285714


### Combine the smaller areas and keep a useful percentage

I keep **area code, familiar local name and households in fuel poverty (%)** in my compact MSOA table. All estimates are for **2024**.

For each MSOA I sum the estimated fuel-poor households and total households in its constituent LSOAs, then calculate **100 × fuel-poor households ÷ total households**. I don’t average the LSOA percentages: a small area and a large area should not contribute equal weight simply because each has one row.

I retain the selected LSOA rows and an MSOA count-support table separately. The workbook records May 2026 geography codes; my bridge uses 2021 LSOA/MSOA codes. The code checks complete coverage and one parent per selected LSOA, rather than assuming compatibility from the filenames. The counts are rounded estimates, so my result is a derived summary, not a newly published official MSOA statistic.

In [17]:
fuel_lsoa_support = fuel_selected.merge(
    fuel_lookup[["LSOA21CD", "MSOA21CD"]],
    left_on="LSOA Code", right_on="LSOA21CD", how="left", validate="one_to_one",
)
assert len(fuel_lsoa_support) == len(fuel_selected)
assert fuel_lsoa_support["MSOA21CD"].notna().all()
fuel_count_fields = ["Number of households", "Number of households in fuel poverty"]
for field in [*fuel_count_fields, "Proportion of households fuel poor (%)"]:
    fuel_lsoa_support[field] = pd.to_numeric(fuel_lsoa_support[field], errors="raise")
assert fuel_lsoa_support[fuel_count_fields].notna().all().all()
assert fuel_lsoa_support[fuel_count_fields].ge(0).all().all()
assert fuel_lsoa_support["Number of households"].gt(0).all()
assert fuel_lsoa_support["Number of households in fuel poverty"].le(fuel_lsoa_support["Number of households"]).all()

# Sum numerators and denominators before calculating the MSOA percentage.
fuel_msoa_support = fuel_lsoa_support.groupby("MSOA21CD", as_index=False)[fuel_count_fields].sum()
fuel_msoa_support = fuel_msoa_support.rename(columns={"MSOA21CD": "area_code"})
fuel_msoa_support["lsoa_count"] = fuel_msoa_support["area_code"].map(fuel_lsoa_support.groupby("MSOA21CD").size())
fuel_display = pd.DataFrame({
    "area_code": fuel_msoa_support["area_code"],
    "local_area_name": fuel_msoa_support["area_code"].map(local_name_by_code),
    "fuel_poor_households_pct": 100 * fuel_msoa_support["Number of households in fuel poverty"] / fuel_msoa_support["Number of households"],
})
assert set(fuel_display["area_code"]) == coverage_codes
assert fuel_display["area_code"].is_unique
assert fuel_display["local_area_name"].notna().all()
assert fuel_display["fuel_poor_households_pct"].between(0, 100).all()
print(f"Compact table: {len(fuel_display):,} MSOAs; {len(fuel_display.columns)} fields")
birmingham_codes = set(area_code for area_code, entry in local_names.items() if entry["authority"] == "Birmingham")
display(fuel_display.loc[fuel_display["area_code"].isin(birmingham_codes)].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 MSOAs; 3 fields


,area_code,local_area_name,fuel_poor_households_pct
0,E02001827,Hill Hook,7.13
1,E02001828,Four Oaks,8.43
2,E02001829,Little Sutton,7.10
3,E02001830,Sutton Coldfield North,7.27
4,E02001831,Reddicap Heath,9.02
5,E02001832,Rectory Park,6.83
6,E02001833,Kingstanding North,10.06
7,E02001834,Old Oscott,13.16
8,E02001835,Sutton Coldfield South,10.28
9,E02001836,New Oscott,8.70


Preview rounded only; saved percentages retain precision.


### Check and save my fuel poverty dataset

I save my compact table in **`data/cleaned/Fuel Poverty/`**, together with separate LSOA and MSOA supporting files. The supporting files preserve the source counts, area relationships and number of LSOAs combined. I back up existing exports before replacing them and verify the saved contents by reading them back.

Fuel poverty is now ready as a contextual candidate. Its dates, estimated counts and small-area uncertainty still matter when I explore its relationship with HLE.

In [18]:
fuel_folder = project_data("Cleaned/Fuel Poverty")
fuel_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("fuel_poverty_2024_msoa.csv", fuel_display),
    ("fuel_poverty_2024_msoa_support.csv", fuel_msoa_support),
    ("fuel_poverty_2024_lsoa_support.csv", fuel_lsoa_support),
]:
    destination = fuel_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Fuel Poverty"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified fuel_poverty_2024_msoa.csv: 660 rows; 3 fields
Saved and verified fuel_poverty_2024_msoa_support.csv: 660 rows; 4 fields
Saved and verified fuel_poverty_2024_lsoa_support.csv: 3,205 rows; 10 fields


## 5. Pollution — understand the grid sources

I have two **2024 annual-mean background pollution estimates**: nitrogen dioxide (**NO₂**) and fine particles (**PM2.5**). They are separate measures, in **micrograms per cubic metre (µg/m³)**. PM2.5 refers to particles no more than 2.5 micrometres in diameter.

Unlike the Census tables, these files describe **1 km × 1 km grid squares**, not statistical areas. Each has four columns: a grid code, the square’s centre coordinates (`x` and `y`) and a pollutant value. The coordinates use the British National Grid. I need a spatial calculation before these become MSOA measures.

I’m interested in whether the pollution pattern is associated with HLE. These are modelled background estimates, not readings at every address or a complete description of roadside conditions. Their 2024 period also follows the 2019–2023 HLE period.

[Defra’s source documentation](https://uk-air.defra.gov.uk/data/pcm-data) explains the grid centres, units and literal `MISSING` markers. I read those markers as missing values, not zero. The first five file rows are introductory text; `skiprows=5` starts at the real header.

In [19]:
no2_raw = pd.read_csv(raw_file("defra_no2_2024_1km.csv"), skiprows=5, na_values=["MISSING"])
pm25_raw = pd.read_csv(raw_file("defra_pm25_2024_1km.csv"), skiprows=5, na_values=["MISSING"])
for label, table, value_field in [("NO2", no2_raw, "no22024"), ("PM2.5", pm25_raw, "pm252024g")]:
    assert table["gridcode"].is_unique
    assert not table.duplicated(["x", "y"]).any()
    table[value_field] = pd.to_numeric(table[value_field], errors="raise")
    print(f"{label}: {len(table):,} grid cells; {len(table.columns)} fields; {table[value_field].isna().sum()} unavailable values nationally")
    display(pd.DataFrame({"Source field": table.columns, "What it contains": [
        "Grid-square identifier — not an MSOA code",
        "Grid centre easting in metres",
        "Grid centre northing in metres",
        "2024 annual-mean concentration in micrograms per cubic metre",
    ]}))

NO2: 254,905 grid cells; 4 fields; 0 unavailable values nationally


,Source field,What it contains
0,gridcode,Grid-square identifier — not an MSOA code
1,x,Grid centre easting in metres
2,y,Grid centre northing in metres
3,no22024,2024 annual-mean concentration in micrograms p...


PM2.5: 254,905 grid cells; 4 fields; 1 unavailable values nationally


,Source field,What it contains
0,gridcode,Grid-square identifier — not an MSOA code
1,x,Grid centre easting in metres
2,y,Grid centre northing in metres
3,pm252024g,2024 annual-mean concentration in micrograms p...


### Select the relevant geography and inspect Birmingham examples

I first keep grid squares that could overlap the bounding extent of my 660 MSOAs. I allow half a kilometre beyond each edge because the coordinates describe square centres, not their corners. This is a **candidate selection**: a rectangle can include squares outside the actual area polygons. The final spatial matching must remove those extras.

For the previews I use grid centres inside Birmingham’s polygons, so the examples relate to the city rather than whichever national records happen to come first. They are not the most polluted locations. I keep the source coordinates at this stage because they are essential to geographical matching.

In [20]:
pollution_areas = area_boundaries.to_crs(27700)
xmin, ymin, xmax, ymax = pollution_areas.total_bounds
pollution_candidates = {}
for label, table in [("NO2", no2_raw), ("PM2.5", pm25_raw)]:
    candidate = table.loc[
        table["x"].between(xmin - 500, xmax + 500)
        & table["y"].between(ymin - 500, ymax + 500)
    ].copy()
    pollution_candidates[label] = candidate
    # Preview city grid centres only; the final MSOA method is still to be chosen.
    points = gpd.GeoDataFrame(candidate, geometry=gpd.points_from_xy(candidate["x"], candidate["y"]), crs=27700)
    city_areas = pollution_areas.loc[pollution_areas["MSOA21CD"].isin(birmingham_codes), ["geometry"]]
    city_points = gpd.sjoin(points, city_areas, how="inner", predicate="within")
    print(f"{label}: {len(candidate):,} candidate grid cells across the coverage extent")
    display(city_points.drop(columns=["geometry", "index_right"]).head(5))

NO2: 14,630 candidate grid cells across the coverage extent


,gridcode,x,y,no22024
168345,689731,410500,300500,10.233360
168346,689732,411500,300500,9.365366
168347,689733,412500,300500,9.100857
168741,690420,409500,299500,10.759030
168742,690421,410500,299500,10.736290


PM2.5: 14,630 candidate grid cells across the coverage extent


,gridcode,x,y,pm252024g
168345,689731,410500,300500,7.289942
168346,689732,411500,300500,7.159149
168347,689733,412500,300500,7.092629
168741,690420,409500,299500,7.392168
168742,690421,410500,299500,7.415641


### Use an area-weighted mean within each MSOA boundary

I favour the **area-weighted mean for the geographical comparison**, and that is the method I’m using. I combine the grid squares overlapping each MSOA, weighting each concentration by the area of its overlap **inside the MSOA boundary**. The part of a square outside that boundary contributes nothing to that MSOA.

I calculate **sum of concentration × overlap area ÷ sum of valid overlap area**. I treat the modelled value as constant within each 1 km square. Small MSOAs may share a grid value; this method cannot create detail finer than the source grid.

This describes average modelled background concentration across the area. It is not population-weighted exposure: fields and residential streets contribute according to their area, not the number of residents.

I check how much of every MSOA is covered by available values and keep those checks separately. If any area has incomplete valid coverage, I flag it rather than silently present a partial mean as complete. All pollutant estimates are for **2024**, in **µg/m³**.

In [21]:
import numpy as np
from shapely.geometry import box

assert pollution_areas.geometry.is_valid.all(), "Invalid MSOA geometry needs review"
assert pollution_areas["MSOA21CD"].is_unique
pollution_tables = {}
pollution_checks = {}
pollution_overlaps = {}

for label, value_field, output_field in [
    ("NO2", "no22024", "no2_area_mean_ug_m3"),
    ("PM2.5", "pm252024g", "pm25_area_mean_ug_m3"),
]:
    candidate = pollution_candidates[label].copy()
    # The supplied coordinates are square centres: construct 1 km cells around them.
    cells = gpd.GeoDataFrame(candidate, geometry=[
        box(x - 500, y - 500, x + 500, y + 500)
        for x, y in zip(candidate["x"], candidate["y"])
    ], crs=27700)
    # Clip cells to MSOA polygons; only the intersection area supplies weight.
    overlap = gpd.overlay(
        pollution_areas[["MSOA21CD", "geometry"]], cells,
        how="intersection", keep_geom_type=True,
    )
    overlap["overlap_m2"] = overlap.geometry.area
    overlap = overlap.loc[overlap["overlap_m2"].gt(0)].copy()
    valid = overlap.loc[overlap[value_field].notna()].copy()
    assert valid[value_field].ge(0).all(), "Negative pollution concentration"
    valid["weighted_concentration"] = valid[value_field] * valid["overlap_m2"]
    sums = valid.groupby("MSOA21CD")[["weighted_concentration", "overlap_m2"]].sum()
    means = sums["weighted_concentration"] / sums["overlap_m2"]

    checks = pollution_areas[["MSOA21CD"]].copy().rename(columns={"MSOA21CD": "area_code"})
    checks["msoa_area_m2"] = pollution_areas.geometry.area.to_numpy()
    checks["grid_overlap_m2"] = checks["area_code"].map(overlap.groupby("MSOA21CD")["overlap_m2"].sum()).fillna(0)
    checks["valid_overlap_m2"] = checks["area_code"].map(sums["overlap_m2"]).fillna(0)
    checks["valid_coverage_pct"] = 100 * checks["valid_overlap_m2"] / checks["msoa_area_m2"]
    checks["intersecting_cells"] = checks["area_code"].map(overlap.groupby("MSOA21CD")["gridcode"].nunique()).fillna(0).astype(int)
    # Allow only a tiny numerical tolerance for geometric calculations.
    assert checks["valid_coverage_pct"].le(100.0001).all(), "Overlapping grid cells counted twice"
    checks["complete_valid_coverage"] = np.isclose(checks["valid_overlap_m2"], checks["msoa_area_m2"], rtol=1e-6, atol=1)
    result = pd.DataFrame({
        "area_code": checks["area_code"],
        "local_area_name": checks["area_code"].map(local_name_by_code),
        output_field: checks["area_code"].map(means),
    })
    # Preserve missingness for any area without complete valid coverage.
    result.loc[~checks["complete_valid_coverage"], output_field] = np.nan
    assert result["area_code"].is_unique and set(result["area_code"]) == coverage_codes
    assert result["local_area_name"].notna().all()
    pollution_tables[label] = result
    pollution_checks[label] = checks
    pollution_overlaps[label] = overlap.drop(columns="geometry")
    print(f"{label}: {len(result):,} MSOAs; {len(result.columns)} compact fields")
    print(f"Complete valid coverage: {checks['complete_valid_coverage'].sum():,}; unavailable area means: {result[output_field].isna().sum()}")
    display(result.loc[result["area_code"].isin(birmingham_codes)].head(10).round(2))

NO2: 660 MSOAs; 3 compact fields
Complete valid coverage: 660; unavailable area means: 0


,area_code,local_area_name,no2_area_mean_ug_m3
0,E02001827,Hill Hook,10.11
1,E02001828,Four Oaks,10.44
2,E02001829,Little Sutton,10.43
3,E02001830,Sutton Coldfield North,11.43
4,E02001831,Reddicap Heath,11.37
5,E02001832,Rectory Park,12.50
6,E02001833,Kingstanding North,13.17
7,E02001834,Old Oscott,14.22
8,E02001835,Sutton Coldfield South,12.81
9,E02001836,New Oscott,13.80


PM2.5: 660 MSOAs; 3 compact fields
Complete valid coverage: 660; unavailable area means: 0


,area_code,local_area_name,pm25_area_mean_ug_m3
0,E02001827,Hill Hook,7.29
1,E02001828,Four Oaks,7.32
2,E02001829,Little Sutton,7.30
3,E02001830,Sutton Coldfield North,7.48
4,E02001831,Reddicap Heath,7.46
5,E02001832,Rectory Park,7.58
6,E02001833,Kingstanding North,7.88
7,E02001834,Old Oscott,8.09
8,E02001835,Sutton Coldfield South,7.68
9,E02001836,New Oscott,7.92


### Save my pollution datasets

I save separate three-column NO₂ and PM2.5 tables in **`data/cleaned/Pollution/`**: area code, familiar local name and the area-weighted concentration. My coverage checks and grid-overlap evidence go into separate supporting files.

The previews show Birmingham examples, rounded for readability. Saved values retain precision. I back up existing files before replacing them, then read each saved file back to check its contents. No source grid file or MSOA boundary is changed.

In [22]:
pollution_folder = project_data("Cleaned/Pollution")
pollution_folder.mkdir(parents=True, exist_ok=True)
for label, stem in [("NO2", "no2_2024"), ("PM2.5", "pm25_2024")]:
    for filename, table in [
        (f"{stem}_msoa.csv", pollution_tables[label]),
        (f"{stem}_msoa_coverage_checks.csv", pollution_checks[label]),
        (f"{stem}_grid_overlap_support.csv", pollution_overlaps[label]),
    ]:
        destination = pollution_folder / filename
        if destination.exists():
            backup_folder = project_folder / "VS_CODE/Backups/Outputs/Pollution"
            backup_folder.mkdir(parents=True, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
            shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
        table.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
        print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified no2_2024_msoa.csv: 660 rows; 3 fields
Saved and verified no2_2024_msoa_coverage_checks.csv: 660 rows; 7 fields
Saved and verified no2_2024_grid_overlap_support.csv: 16,312 rows; 6 fields
Saved and verified pm25_2024_msoa.csv: 660 rows; 3 fields
Saved and verified pm25_2024_msoa_coverage_checks.csv: 660 rows; 7 fields
Saved and verified pm25_2024_grid_overlap_support.csv: 16,312 rows; 6 fields


## 6. Economic activity — understand the source

I now prepare **Census 2021 TS066: economic activity status**. It describes residents aged **16 and over**, including employment, unemployment and economic inactivity.

This is useful context for asking whether HLE patterns are associated with participation in work and the labour market. It does not tell me whether jobs are secure, well paid or hazardous. The 16+ population also includes retired people, so an employment share is not a working-age employment rate.

My process remains **select geography → inspect fields → choose measures → check and save**. I read the original MSOA CSV from its ZIP, then select the same 660 codes as HLE. I keep all source columns for inspection.

In [23]:
activity_zip = raw_file("census2021-ts066.zip")
with ZipFile(activity_zip) as archive:
    with archive.open("census2021-ts066-msoa.csv") as source_file:
        activity_raw = pd.read_csv(source_file)
assert activity_raw["geography code"].is_unique
assert activity_raw["date"].astype(str).eq("2021").all()
activity_support = activity_raw.loc[activity_raw["geography code"].isin(coverage_codes)].copy()
assert set(activity_support["geography code"]) == coverage_codes
print(f"National source: {len(activity_raw):,} areas")
print(f"Selected coverage: {len(activity_support):,} areas; {len(activity_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 34 fields


### Inspect all 34 fields before thinning the table

This source has a lot of branches. I show the complete field list and a sideways sample of five Birmingham areas in source order. They are examples, not a ranking.

The counts have nested categories: employment contains employee and self-employed branches, which in turn contain full-time and part-time branches. **I must not add parent totals to their children**, or I’ll count people more than once.

The table also separates economically active full-time students from other economically active residents. Students can be employed or unemployed too; I need both branches for measures that include all residents.

In [24]:
activity_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original statistical area name",
    "geography code": "MSOA code — link to HLE and the map",
}
display(pd.DataFrame({
    "Source field": activity_support.columns,
    "What it contains": [activity_identifiers.get(field, "Count of residents aged 16+: " + field.removeprefix("Economic activity status: ")) for field in activity_support.columns],
}))
birmingham_activity = activity_support.loc[activity_support["geography"].str.startswith("Birmingham", na=False)]
# Turn the preview sideways so all the long field names remain visible.
activity_preview = birmingham_activity.head(5).set_index("geography code").T
activity_preview.index.name = "Source field"
display(activity_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Economic activity status: Total: All usual res...,Count of residents aged 16+: Total: All usual ...
4,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
5,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
6,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
7,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
8,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...
9,Economic activity status: Economically active ...,Count of residents aged 16+: Economically acti...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Economic activity status: Total: All usual residents aged 16 years and over,5188,5859,7139,8122,8333
Economic activity status: Economically active (excluding full-time students),2877,3173,4403,4516,4946
Economic activity status: Economically active (excluding full-time students):In employment,2763,3048,4259,4396,4650
Economic activity status: Economically active (excluding full-time students):In employment:Employee,2278,2484,3561,3497,4043
Economic activity status: Economically active (excluding full-time students): In employment: Employee: Part-time,555,666,888,884,1020
Economic activity status: Economically active (excluding full-time students): In employment: Employee: Full-time,1723,1818,2673,2613,3023
Economic activity status: Economically active (excluding full-time students):In employment:Self-employed with employees,99,139,152,259,73


### Keep two clearly defined measures

More than any other table so far, this one shows why I need to examine and explore a dataset before deciding what to keep. It is wide, deep and measuring a lot of different things—taking every column would give me a forest of numbers rather than a clear view of the question. My aim, as throughout this project, is to select the key elements that might reveal useful patterns when I apply machine learning to comparisons, groupings or classifications. I’m choosing relevant, clearly defined measures to investigate, not assuming in advance which ones will prove informative.

I start with two candidate measures:

- **Employed residents aged 16+ (%)**: employed residents, including employed full-time students, divided by all residents aged 16+.
- **Unemployment rate among economically active residents (%)**: unemployed residents, including unemployed full-time students, divided by all economically active residents.

These percentages use **different denominators** and are not complements. Economically inactive residents are outside the unemployment denominator: retirement, study, caring and illness are not interchangeable with unemployment.

My compact table has **area code, familiar local name and these two percentages**. The period is **Census 2021**, stated here rather than repeated as a column. I leave the detailed employment and inactivity breakdowns in the supporting file for later review. If I later examine long-term sickness or disability, I will account for its overlap with the health question itself.

In [25]:
activity_total_field = "Economic activity status: Total: All usual residents aged 16 years and over"
active_branches = [
    "Economic activity status: Economically active (excluding full-time students)",
    "Economic activity status: Economically active and a full-time student",
]
employment_fields = [branch + ":In employment" for branch in active_branches]
unemployment_fields = [branch + ": Unemployed" for branch in active_branches]
needed_activity_fields = [activity_total_field, *active_branches, *employment_fields, *unemployment_fields]
activity_counts = activity_support[needed_activity_fields]
assert activity_counts.notna().all().all(), "Missing counts need review"
assert activity_counts.ge(0).all().all(), "Negative resident counts"

# Include students consistently in both the numerators and relevant denominators.
residents_16plus = activity_counts[activity_total_field]
active_residents = activity_counts[active_branches].sum(axis=1)
employed_residents = activity_counts[employment_fields].sum(axis=1)
unemployed_residents = activity_counts[unemployment_fields].sum(axis=1)
assert residents_16plus.gt(0).all() and active_residents.gt(0).all()
assert active_residents.le(residents_16plus).all()
assert employed_residents.le(residents_16plus).all()
assert unemployed_residents.le(active_residents).all()

activity_display = pd.DataFrame({
    "area_code": activity_support["geography code"],
    "local_area_name": activity_support["geography code"].map(local_name_by_code),
    "employed_residents_16plus_pct": 100 * employed_residents / residents_16plus,
    "unemployment_rate_active_pct": 100 * unemployed_residents / active_residents,
})
assert activity_display["area_code"].is_unique
assert activity_display["local_area_name"].notna().all()
assert activity_display[["employed_residents_16plus_pct", "unemployment_rate_active_pct"]].ge(0).all().all()
assert activity_display[["employed_residents_16plus_pct", "unemployment_rate_active_pct"]].le(100).all().all()
print(f"Compact table: {len(activity_display):,} areas; {len(activity_display.columns)} fields")
display(activity_display.loc[activity_display["area_code"].isin(birmingham_activity["geography code"])].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 areas; 4 fields


,area_code,local_area_name,employed_residents_16plus_pct,unemployment_rate_active_pct
1746,E02001827,Hill Hook,54.59,4.90
1747,E02001828,Four Oaks,52.94,4.82
1748,E02001829,Little Sutton,60.71,3.67
1749,E02001830,Sutton Coldfield North,55.13,3.24
1750,E02001831,Reddicap Heath,57.16,6.53
1751,E02001832,Rectory Park,58.02,4.41
1752,E02001833,Kingstanding North,58.99,5.15
1753,E02001834,Old Oscott,56.78,8.08
1754,E02001835,Sutton Coldfield South,60.80,4.23
1755,E02001836,New Oscott,56.62,4.39


Preview rounded only; saved percentages retain precision.


### Check and save my economic activity dataset

I save the four-column table in **`data/cleaned/Economic Activity/`** and preserve the 34-field source-count table separately. I keep timestamped backups before replacing existing exports. I read the CSVs back to verify their contents.

I now have comparable area-level measures with explicit populations and denominators. I haven’t joined them to HLE or tested their associations yet.

In [26]:
activity_folder = project_data("Cleaned/Economic Activity")
activity_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("economic_activity_2021_msoa.csv", activity_display),
    ("economic_activity_2021_msoa_support.csv", activity_support),
]:
    destination = activity_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Economic Activity"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified economic_activity_2021_msoa.csv: 660 rows; 4 fields
Saved and verified economic_activity_2021_msoa_support.csv: 660 rows; 34 fields


## 7. Qualifications — understand the source

The last dataset in my original preparation sequence is **Census 2021 TS067: highest level of qualification**. It describes usual residents aged **16 and over**, not school performance or the quality of local schools.

I want to investigate whether educational attainment patterns are associated with the HLE differences on my map and with the contextual groups I may find through machine learning. Qualifications can describe one part of an area’s social profile; they do not establish an individual’s health or explain an HLE difference by themselves.

I follow the same process: **select geography → inspect all fields → choose useful measures → check and save**. I read the MSOA CSV directly from the source ZIP and keep the same 660 area codes as HLE.

In [27]:
qualifications_zip = raw_file("census2021-ts067.zip")
with ZipFile(qualifications_zip) as archive:
    with archive.open("census2021-ts067-msoa.csv") as source_file:
        qualifications_raw = pd.read_csv(source_file)
assert qualifications_raw["geography code"].is_unique
assert qualifications_raw["date"].astype(str).eq("2021").all()
qualifications_support = qualifications_raw.loc[
    qualifications_raw["geography code"].isin(coverage_codes)
].copy()
assert set(qualifications_support["geography code"]) == coverage_codes
print(f"National source: {len(qualifications_raw):,} areas")
print(f"Selected coverage: {len(qualifications_support):,} areas; {len(qualifications_support.columns)} fields")

National source: 7,264 areas
Selected coverage: 660 areas; 11 fields


### Inspect the fields before selecting the highlights

I show all source fields and a sideways sample of five Birmingham areas. The examples follow source order, not a ranking of attainment.

The qualification categories contain counts of residents, so larger areas may have larger counts without a different educational profile. Percentages using this table’s **16+ resident total** let me compare the composition of differently sized areas.

In [28]:
qualification_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original statistical area name",
    "geography code": "MSOA code — link to HLE and the map",
}
display(pd.DataFrame({
    "Source field": qualifications_support.columns,
    "What it contains": [qualification_identifiers.get(field, "Resident count: " + field) for field in qualifications_support.columns],
}))
birmingham_qualifications = qualifications_support.loc[
    qualifications_support["geography"].str.startswith("Birmingham", na=False)
]
# A vertical preview keeps the long category labels readable.
qualifications_preview = birmingham_qualifications.head(5).set_index("geography code").T
qualifications_preview.index.name = "Source field"
display(qualifications_preview)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original statistical area name
2,geography code,MSOA code — link to HLE and the map
3,Highest level of qualification: Total: All usu...,Resident count: Highest level of qualification...
4,Highest level of qualification: No qualifications,Resident count: Highest level of qualification...
5,Highest level of qualification: Level 1 and en...,Resident count: Highest level of qualification...
6,Highest level of qualification: Level 2 qualif...,Resident count: Highest level of qualification...
7,Highest level of qualification: Apprenticeship,Resident count: Highest level of qualification...
8,Highest level of qualification: Level 3 qualif...,Resident count: Highest level of qualification...
9,Highest level of qualification: Level 4 qualif...,Resident count: Highest level of qualification...


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
Source field,,,,,
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Highest level of qualification: Total: All usual residents aged 16 years and over,5184,5859,7134,8123,8332
Highest level of qualification: No qualifications,728,923,750,906,1641
Highest level of qualification: Level 1 and entry level qualifications,428,469,526,585,878
Highest level of qualification: Level 2 qualifications,718,784,945,915,1183
Highest level of qualification: Apprenticeship,215,248,314,304,431
Highest level of qualification: Level 3 qualifications,810,860,1146,1228,1434
Highest level of qualification: Level 4 qualifications and above,2165,2417,3302,4016,2558


### Keep two useful qualification measures (actually three)

I started with no qualifications and Level 4+, but that only showed the lower and higher ends of the qualification profile. I want a view of the middle as well, so I’m adding **Level 2**. My selection is **0, 2 and 4+**: three recognisable points across the distribution, without carrying every category into the compact table.

- **No qualifications (%) — the lower end:** residents reporting no qualifications, divided by all residents aged 16+.
- **Level 2 qualifications (%) — the middle category:** residents whose highest qualification is in the Census Level 2 category, broadly five or more GCSE passes at grades 4–9 or equivalents, divided by the same 16+ total.
- **Level 4 qualifications and above (%) — the higher end:** residents in the source’s grouped higher-qualification category, from HNC-level qualifications through degrees and postgraduate qualifications, divided by the same total.

I’m using **Level 2 itself, not Level 2 or above**. That gives me the middle category rather than a broad measure that also includes the higher category I already have. Level 4+ is the highest band available in this table, not a numerical maximum qualification score. These are three percentages, not a minimum, mean and maximum or a calculated qualification index.

The three shares do **not** add up to 100%: Level 1, apprenticeships, Level 3 and other qualifications remain outside this selection. I retain those counts in my supporting file if the analysis suggests I should explore the full profile later.

My compact table now has **five columns: area code, familiar local name and the three percentages**. All observations are from **Census 2021**. Adding the middle gives me a broader view to investigate; it does not establish which measures will relate to HLE.

Source definitions: [ONS highest level of qualification](https://www.ons.gov.uk/census/census2021dictionary/variablesbytopic/educationvariablescensus2021/highestlevelofqualification).

In [29]:
qualification_total = next(field for field in qualifications_support if ": Total:" in field)
no_qualification_field = next(field for field in qualifications_support if field.endswith(": No qualifications"))
level4_field = next(field for field in qualifications_support if field.endswith(": Level 4 qualifications and above"))
level2_field = next(field for field in qualifications_support if field.endswith(": Level 2 qualifications"))
qualification_counts = qualifications_support[[qualification_total, no_qualification_field, level2_field, level4_field]]
assert qualification_counts.notna().all().all(), "Missing counts need review"
assert qualification_counts.ge(0).all().all(), "Negative resident counts"
assert qualification_counts[qualification_total].gt(0).all(), "Invalid resident denominator"
# Use this table's own population total for both percentages.
for field in [no_qualification_field, level2_field, level4_field]:
    assert qualification_counts[field].le(qualification_counts[qualification_total]).all()
assert (qualification_counts[no_qualification_field] + qualification_counts[level2_field] + qualification_counts[level4_field]).le(qualification_counts[qualification_total]).all()
qualifications_display = pd.DataFrame({
    "area_code": qualifications_support["geography code"],
    "local_area_name": qualifications_support["geography code"].map(local_name_by_code),
    "no_qualifications_pct": 100 * qualification_counts[no_qualification_field] / qualification_counts[qualification_total],
    "level2_qualifications_pct": 100 * qualification_counts[level2_field] / qualification_counts[qualification_total],
    "level4plus_pct": 100 * qualification_counts[level4_field] / qualification_counts[qualification_total],
})
assert qualifications_display["area_code"].is_unique
assert qualifications_display["local_area_name"].notna().all()
for field in ["no_qualifications_pct", "level2_qualifications_pct", "level4plus_pct"]:
    assert qualifications_display[field].between(0, 100).all()
print(f"Compact table: {len(qualifications_display):,} areas; {len(qualifications_display.columns)} fields")
display(qualifications_display.loc[
    qualifications_display["area_code"].isin(birmingham_qualifications["geography code"])
].head(10).round(2))
print("Preview rounded only; saved percentages retain precision.")

Compact table: 660 areas; 5 fields


,area_code,local_area_name,no_qualifications_pct,level2_qualifications_pct,level4plus_pct
1746,E02001827,Hill Hook,14.04,13.85,41.76
1747,E02001828,Four Oaks,15.75,13.38,41.25
1748,E02001829,Little Sutton,10.51,13.25,46.29
1749,E02001830,Sutton Coldfield North,11.15,11.26,49.44
1750,E02001831,Reddicap Heath,19.70,14.20,30.70
1751,E02001832,Rectory Park,14.05,12.92,41.94
1752,E02001833,Kingstanding North,17.55,13.58,32.21
1753,E02001834,Old Oscott,25.39,15.36,20.79
1754,E02001835,Sutton Coldfield South,13.30,12.76,46.91
1755,E02001836,New Oscott,15.48,11.85,41.70


Preview rounded only; saved percentages retain precision.


### Check and save my qualifications dataset

I save the five-column compact table in **`data/cleaned/Qualifications/`**, with the original 11-field source-count table in a separate supporting file. I back up existing exports before replacement and read the saved CSVs back to verify their contents.

This completes my original preparation sequence: **HLE, household deprivation, overcrowding, fuel poverty, pollution, economic activity and qualifications**. Each compact dataset uses the same MSOA codes and familiar names. Age remains a possible later addition, not part of this completed sequence. I have not yet joined these datasets or tested their associations.

In [30]:
qualifications_folder = project_data("Cleaned/Qualifications")
qualifications_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("qualifications_2021_msoa.csv", qualifications_display),
    ("qualifications_2021_msoa_support.csv", qualifications_support),
]:
    destination = qualifications_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Qualifications"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified qualifications_2021_msoa.csv: 660 rows; 5 fields
Saved and verified qualifications_2021_msoa_support.csv: 660 rows; 11 fields


## 8. Household income — understand the source

I am adding household income as financial context for the HLE differences on my map. I want a useful measure of household resources, rather than four versions of much the same story.

The ONS workbook covers **April 2022 to March 2023 (financial year ending 2023)** and contains four modelled MSOA averages: total income, net income, and household-adjusted net income before and after housing costs. These are annual household measures, not individual salaries or a measure of debt.

I follow the same process: **select geography → inspect the fields → keep a useful measure → check and save**. I use the same 660 MSOA codes as HLE.

In [31]:
income_workbook = raw_file("ons_small_area_income_fye2023.xlsx")
income_sheets = [
    "Total annual income", "Net annual income",
    "Net income before housing costs", "Net income after housing costs",
]
display(pd.DataFrame({
    "Source sheet": income_sheets,
    "Measure": [
        "Gross annual household income, including benefits",
        "Disposable annual household income after specified deductions",
        "Disposable income adjusted for household size/composition, before housing costs",
        "Disposable income adjusted for household size/composition, after housing costs",
    ],
}))
# The workbook has three introductory rows above the column headings.
income_raw = pd.read_excel(income_workbook, sheet_name="Net income after housing costs", header=3)
assert income_raw["MSOA code"].is_unique
income_support = income_raw.loc[income_raw["MSOA code"].isin(coverage_codes)].copy()
assert set(income_support["MSOA code"]) == coverage_codes
print(f"National source: {len(income_raw):,} MSOAs")
print(f"Selected coverage: {len(income_support):,} MSOAs; {len(income_support.columns)} fields")

,Source sheet,Measure
0,Total annual income,"Gross annual household income, including benefits"
1,Net annual income,Disposable annual household income after speci...
2,Net income before housing costs,Disposable income adjusted for household size/...
3,Net income after housing costs,Disposable income adjusted for household size/...


National source: 7,264 MSOAs
Selected coverage: 660 MSOAs; 10 fields


### Inspect all ten fields

I show every field in the selected table, followed by five Birmingham examples. These follow source order; they are not a ranking of income.

The confidence limits describe uncertainty around the modelled average. I retain them in the supporting table even though they do not need to crowd the compact feature table.

In [32]:
income_field_notes = [
    "MSOA join key", "Original ONS area name",
    "Local authority identifier", "Local authority name",
    "Region identifier", "Region name",
    "Estimated annual equivalised disposable household income after housing costs (£)",
    "Upper 95% confidence limit (£)", "Lower 95% confidence limit (£)",
    "Width of the confidence interval (£)",
]
assert len(income_field_notes) == len(income_support.columns)
display(pd.DataFrame({"Source field": income_support.columns, "What it contains": income_field_notes}))
birmingham_income = income_support.loc[income_support["Local authority name"].eq("Birmingham")]
display(birmingham_income.head(5).set_index("MSOA code").T)

,Source field,What it contains
0,MSOA code,MSOA join key
1,MSOA name,Original ONS area name
2,Local authority code,Local authority identifier
3,Local authority name,Local authority name
4,Region code,Region identifier
5,Region name,Region name
6,Disposable (net) annual income after housing c...,Estimated annual equivalised disposable househ...
7,Upper confidence limit (£),Upper 95% confidence limit (£)
8,Lower confidence limit (£),Lower 95% confidence limit (£)
9,Confidence interval (£),Width of the confidence interval (£)


MSOA code,E02001827,E02001828,E02001829,E02001830,E02001831
MSOA name,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Local authority code,E08000025,E08000025,E08000025,E08000025,E08000025
Local authority name,Birmingham,Birmingham,Birmingham,Birmingham,Birmingham
Region code,E12000005,E12000005,E12000005,E12000005,E12000005
Region name,West Midlands,West Midlands,West Midlands,West Midlands,West Midlands
Disposable (net) annual income after housing costs (£),36785,36596,37480,43436,32675
Upper confidence limit (£),42015,41801,42953,49849,37451
Lower confidence limit (£),32205,32038,32705,37847,28507
Confidence interval (£),9810,9763,10248,12002,8944


### Keep one useful income measure

I choose **net household income after housing costs, adjusted for household size and composition**. ONS calls this adjustment *equivalisation*: £30,000 supporting one person is a different proposition from £30,000 supporting a family. The adjustment puts households on a more comparable footing.

This measure accounts for specified housing expenses, including rent and mortgage interest; it is not income after every household bill, and mortgage capital repayments are not deducted. It gives me a useful view of household resources after a major expense.

My compact table has **area code, familiar local area name and annual equivalised income after housing costs (£)**. I replace the numbered ONS names with the same familiar names used for HLE. The period is stated above rather than repeated in a column.

These are **modelled area averages**, not the income of every household or proof that income explains HLE. I will investigate the association later. Source definitions: [ONS income estimates, financial year ending 2023](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/bulletins/smallareamodelbasedincomeestimates/financialyearending2023).

In [33]:
income_value_field = "Disposable (net) annual income after housing costs (£)"
income_numeric_fields = [income_value_field, "Lower confidence limit (£)", "Upper confidence limit (£)", "Confidence interval (£)"]
assert income_support[income_numeric_fields].notna().all().all()
assert income_support[income_numeric_fields].ge(0).all().all()
assert income_support["Lower confidence limit (£)"].le(income_support[income_value_field]).all()
assert income_support[income_value_field].le(income_support["Upper confidence limit (£)"]).all()
assert (income_support["Upper confidence limit (£)"] - income_support["Lower confidence limit (£)"]).eq(income_support["Confidence interval (£)"]).all()
income_display = pd.DataFrame({
    "area_code": income_support["MSOA code"],
    "local_area_name": income_support["MSOA code"].map(local_name_by_code),
    "annual_equivalised_income_after_housing_costs_gbp": income_support[income_value_field],
})
assert income_display["area_code"].is_unique
assert income_display["local_area_name"].notna().all()
print(f"Compact table: {len(income_display):,} areas; {len(income_display.columns)} fields")
display(income_display.loc[income_display["area_code"].isin(birmingham_income["MSOA code"])].head(10))

Compact table: 660 areas; 3 fields


,area_code,local_area_name,annual_equivalised_income_after_housing_costs_gbp
2922,E02001827,Hill Hook,36785
2923,E02001828,Four Oaks,36596
2924,E02001829,Little Sutton,37480
2925,E02001830,Sutton Coldfield North,43436
2926,E02001831,Reddicap Heath,32675
2927,E02001832,Rectory Park,37801
2928,E02001833,Kingstanding North,34826
2929,E02001834,Old Oscott,26665
2930,E02001835,Sutton Coldfield South,35867
2931,E02001836,New Oscott,37634


### Check and save my income dataset

I save the three-column table in **`data/cleaned/Income/`**, with the original ten-field selected table in a separate supporting file. The raw workbook still contains all four income measures if I need to revisit the choice.

I back up existing exports before replacement and read the CSVs back to verify them. Income is now ready for later comparison with HLE; I have not joined it to HLE or tested associations yet.

In [34]:
income_folder = project_data("Cleaned/Income")
income_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("income_after_housing_costs_fye2023_msoa.csv", income_display),
    ("income_after_housing_costs_fye2023_msoa_support.csv", income_support),
]:
    destination = income_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Income"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified income_after_housing_costs_fye2023_msoa.csv: 660 rows; 3 fields
Saved and verified income_after_housing_costs_fye2023_msoa_support.csv: 660 rows; 10 fields


## 9. Ethnicity — explore cultural context

I am investigating cultural factors alongside socioeconomic factors as I explore HLE differences. Ethnicity provides one aspect of that context, but a broad bucket such as “Asian” hides distinctions between Chinese, Bangladeshi, Pakistani and Indian residents. I keep the detailed categories rather than squash them together before I have even looked.

This is **Census 2021 TS021**, recording self-reported ethnic group on **21 March 2021**, within the HLE period of 2019–2023. It does not measure religion, beliefs or behaviour: I cannot infer Muslim, Hindu or Sikh populations from ethnic group counts. Religion would require its own dataset.

I initially sought the safeguarded modelled ethnicity data through GeoDS at University College London. The application was not completed because my course co-ordinator declined to act as the academic supervisor required by the application. I therefore use this open Census snapshot. It is worth examining; whether it adds useful analytical context remains to be tested.

I follow the same process: **select geography → inspect all fields → choose measures → check and save**.

In [35]:
ethnicity_zip = raw_file("census2021-ts021.zip")
with ZipFile(ethnicity_zip) as archive:
    with archive.open("census2021-ts021-msoa.csv") as source_file:
        ethnicity_raw = pd.read_csv(source_file)
assert ethnicity_raw["geography code"].is_unique
assert ethnicity_raw["date"].astype(str).eq("2021").all()
ethnicity_support = ethnicity_raw.loc[ethnicity_raw["geography code"].isin(coverage_codes)].copy()
assert set(ethnicity_support["geography code"]) == coverage_codes
print(f"National source: {len(ethnicity_raw):,} MSOAs")
print(f"Selected coverage: {len(ethnicity_support):,} MSOAs; {len(ethnicity_support.columns)} fields")

National source: 7,264 MSOAs
Selected coverage: 660 MSOAs; 28 fields


### Inspect the categories before narrowing the table

I show all 28 fields, then five Birmingham examples sideways so the labels remain readable. The source includes a resident total, five broad-group counts and 19 detailed-category counts, plus three identifiers. The broad groups are subtotals of the detailed categories; I must not add both together and count residents twice.

These examples follow source order, not a ranking. Larger counts can simply mean a larger population, so percentages will make area composition easier to compare.

In [36]:
ethnicity_identifiers = {
    "date": "Census year — 2021 throughout",
    "geography": "Original ONS area name",
    "geography code": "MSOA join key",
}
display(pd.DataFrame({
    "Source field": ethnicity_support.columns,
    "What it contains": [
        ethnicity_identifiers.get(field,
            "Total resident count" if field == "Ethnic group: Total: All usual residents"
            else "Broad-group subtotal" if field.count(":") == 1 else "Detailed-category resident count")
        for field in ethnicity_support.columns
    ],
}))
birmingham_ethnicity = ethnicity_support.loc[ethnicity_support["geography"].str.startswith("Birmingham", na=False)]
display(birmingham_ethnicity.head(5).set_index("geography code").T)

,Source field,What it contains
0,date,Census year — 2021 throughout
1,geography,Original ONS area name
2,geography code,MSOA join key
3,Ethnic group: Total: All usual residents,Total resident count
4,"Ethnic group: Asian, Asian British or Asian Welsh",Broad-group subtotal
5,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
6,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
7,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
8,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count
9,"Ethnic group: Asian, Asian British or Asian We...",Detailed-category resident count


geography code,E02001827,E02001828,E02001829,E02001830,E02001831
date,2021,2021,2021,2021,2021
geography,Birmingham 001,Birmingham 002,Birmingham 003,Birmingham 004,Birmingham 005
Ethnic group: Total: All usual residents,6450,7141,9160,9686,10682
"Ethnic group: Asian, Asian British or Asian Welsh",850,653,1160,1374,819
"Ethnic group: Asian, Asian British or Asian Welsh: Bangladeshi",24,20,38,48,43
"Ethnic group: Asian, Asian British or Asian Welsh: Chinese",78,80,142,102,87
"Ethnic group: Asian, Asian British or Asian Welsh: Indian",517,359,723,927,376
"Ethnic group: Asian, Asian British or Asian Welsh: Pakistani",126,126,177,220,190
"Ethnic group: Asian, Asian British or Asian Welsh: Other Asian",105,68,80,77,123
"Ethnic group: Black, Black British, Black Welsh, Caribbean or African",134,134,164,184,495


### Keep the detail as percentages

I keep the **19 detailed categories**, each divided by the source’s total usual-resident population. My compact table has **21 columns: area code, familiar local name and 19 percentages**. It is wider than the income table because the distinctions are the point of this exercise.

I leave the five broad subtotals out of the feature table because they repeat information already present in the detailed categories. All original counts remain in the supporting file. The familiar area names match HLE, and the Census date is stated above rather than repeated in every row.

These shares describe an area’s population, not a single culture shared by its residents. I will not label any category as explaining HLE before analysis. Census disclosure protection can create small differences between published totals and category sums; I use the published total without silently changing the counts or forcing the percentages to add to exactly 100%.

Source definitions: [ONS ethnic group, Census 2021](https://www.ons.gov.uk/datasets/TS021/editions/2021/versions/3).

In [37]:
ethnicity_total_field = "Ethnic group: Total: All usual residents"
# Select the leaf categories, excluding both the total and broad subtotals.
ethnicity_detail_fields = [
    field for field in ethnicity_support
    if field.startswith("Ethnic group:") and field.count(":") == 2 and field != ethnicity_total_field
]
assert len(ethnicity_detail_fields) == 19
ethnicity_feature_names = [
    "asian_bangladeshi_pct", "asian_chinese_pct", "asian_indian_pct",
    "asian_pakistani_pct", "asian_other_pct", "black_african_pct",
    "black_caribbean_pct", "black_other_pct", "mixed_white_asian_pct",
    "mixed_white_black_african_pct", "mixed_white_black_caribbean_pct",
    "mixed_other_pct", "white_british_pct", "white_irish_pct",
    "white_gypsy_irish_traveller_pct", "white_roma_pct", "white_other_pct",
    "other_arab_pct", "other_ethnic_group_pct",
]
ethnicity_counts = ethnicity_support[[ethnicity_total_field, *ethnicity_detail_fields]]
assert ethnicity_counts.notna().all().all()
assert ethnicity_counts.ge(0).all().all()
assert ethnicity_counts[ethnicity_total_field].gt(0).all()
ethnicity_display = pd.DataFrame({
    "area_code": ethnicity_support["geography code"],
    "local_area_name": ethnicity_support["geography code"].map(local_name_by_code),
})
for source_field, feature_name in zip(ethnicity_detail_fields, ethnicity_feature_names, strict=True):
    assert ethnicity_counts[source_field].le(ethnicity_counts[ethnicity_total_field]).all()
    ethnicity_display[feature_name] = 100 * ethnicity_counts[source_field] / ethnicity_counts[ethnicity_total_field]
assert ethnicity_display["area_code"].is_unique
assert ethnicity_display["local_area_name"].notna().all()
assert ethnicity_display[ethnicity_feature_names].ge(0).all().all()
assert ethnicity_display[ethnicity_feature_names].le(100).all().all()
# Keep a dictionary so concise CSV column names remain traceable to ONS categories.
ethnicity_dictionary = pd.DataFrame({
    "feature": ethnicity_feature_names,
    "source_field": ethnicity_detail_fields,
    "unit": "percent of all usual residents",
})
display(ethnicity_dictionary)
print(f"Compact table: {len(ethnicity_display):,} areas; {len(ethnicity_display.columns)} fields")
ethnicity_example = ethnicity_display.loc[ethnicity_display["area_code"].isin(birmingham_ethnicity["geography code"])].head(5)
ethnicity_preview = ethnicity_example.set_index("local_area_name").T
ethnicity_preview.columns.name = "Local area name →"
# Shade whole area columns, including their headers, to guide the eye downwards.
from IPython.display import HTML
preview_css = """<style>
.ethnicity-column-preview th, .ethnicity-column-preview td {
    color: #17212b !important; padding: 6px 12px;
    border-bottom: 1px solid #bdc8d1; text-align: right;
}
.ethnicity-column-preview th { background: #f2f4f7 !important; text-align: left; }
.ethnicity-column-preview tr > :nth-child(2n+2) { background: #dceefa !important; }
.ethnicity-column-preview tr > :nth-child(2n+3) { background: #fff0d2 !important; }
.ethnicity-column-preview thead th { white-space: nowrap; }
</style>"""
preview_html = ethnicity_preview.to_html(
    classes="ethnicity-column-preview", float_format=lambda value: f"{value:.2f}", border=0
)
display(HTML(preview_css + preview_html))
ethnicity_sum_difference = ethnicity_counts[ethnicity_detail_fields].sum(axis=1) - ethnicity_counts[ethnicity_total_field]
print(f"Largest absolute difference between detailed counts and published total: {ethnicity_sum_difference.abs().max():,} residents")
print("Preview rounded only; saved percentages retain precision.")

,feature,source_field,unit
0,asian_bangladeshi_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
1,asian_chinese_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
2,asian_indian_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
3,asian_pakistani_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
4,asian_other_pct,"Ethnic group: Asian, Asian British or Asian We...",percent of all usual residents
5,black_african_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
6,black_caribbean_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
7,black_other_pct,"Ethnic group: Black, Black British, Black Wels...",percent of all usual residents
8,mixed_white_asian_pct,Ethnic group: Mixed or Multiple ethnic groups:...,percent of all usual residents
9,mixed_white_black_african_pct,Ethnic group: Mixed or Multiple ethnic groups:...,percent of all usual residents


Compact table: 660 areas; 21 fields


Local area name →,Hill Hook,Four Oaks,Little Sutton,Sutton Coldfield North,Reddicap Heath
area_code,E02001827,E02001828,E02001829,E02001830,E02001831
asian_bangladeshi_pct,0.37,0.28,0.41,0.50,0.40
asian_chinese_pct,1.21,1.12,1.55,1.05,0.81
asian_indian_pct,8.02,5.03,7.89,9.57,3.52
asian_pakistani_pct,1.95,1.76,1.93,2.27,1.78
asian_other_pct,1.63,0.95,0.87,0.79,1.15
black_african_pct,0.78,0.66,0.68,0.71,1.59
black_caribbean_pct,0.99,0.99,0.88,0.84,2.34
black_other_pct,0.31,0.22,0.23,0.35,0.70
mixed_white_asian_pct,1.35,0.98,0.75,1.01,1.04


Largest absolute difference between detailed counts and published total: 0 residents
Preview rounded only; saved percentages retain precision.


### Check and save my ethnicity dataset

I save the 21-column table in **`data/cleaned/Ethnicity/`**, alongside the 28-field selected source-count table and a column dictionary. I keep timestamped backups before replacing existing exports, and I read every CSV back to check its contents.

I can now explore these percentages. They are related shares of one population, so I will account for that when I choose the inputs for machine learning. I have not joined them to HLE or tested their usefulness yet.

In [38]:
ethnicity_folder = project_data("Cleaned/Ethnicity")
ethnicity_folder.mkdir(parents=True, exist_ok=True)
for filename, table in [
    ("ethnicity_2021_msoa.csv", ethnicity_display),
    ("ethnicity_2021_msoa_support.csv", ethnicity_support),
    ("ethnicity_2021_feature_dictionary.csv", ethnicity_dictionary),
]:
    destination = ethnicity_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Ethnicity"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True))
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified ethnicity_2021_msoa.csv: 660 rows; 21 fields
Saved and verified ethnicity_2021_msoa_support.csv: 660 rows; 28 fields
Saved and verified ethnicity_2021_feature_dictionary.csv: 19 rows; 3 fields


## 10. Recorded crime — understand the source

I am adding recorded crime as context about the local environment. The source contains **January–December 2024** street-level crime and anti-social behaviour records from seven police forces, including British Transport Police. Locations are anonymised, and the published **2021 LSOA codes** describe geography, not the year of the incidents.

I use the existing LSOA-to-MSOA lookup to select the same 660 areas as HLE, inspect the fields and categories, then calculate rates. **2024 is later than HLE’s 2019–2023 period**, so this is later contextual evidence, I cannot treat it as evidence from the same period as those HLE estimates.

Source definitions and limitations: [Police open data](https://data.police.uk/about/). Reporting changes: [Police changelog](https://data.police.uk/changelog/).

In [39]:
crime_zip = raw_file("police_recorded_crime_2024_regional_forces.zip")
with ZipFile(crime_zip) as archive:
    crime_members = sorted(name for name in archive.namelist() if name.endswith("-street.csv"))
    crime_frames = []
    for name in crime_members:
        with archive.open(name) as source_file:
            frame = pd.read_csv(source_file, dtype=str)
        frame["source_file"] = name
        crime_frames.append(frame)
crime_raw = pd.concat(crime_frames, ignore_index=True)
expected_months = {f"2024-{month:02d}" for month in range(1, 13)}
assert len(crime_members) == 84
assert set(crime_raw["Month"]) == expected_months
crime_lookup = pd.read_csv(project_data("Prepared/Geography/ons_lsoa21_to_msoa21_lad22_exploratory_west_midlands.csv"))
assert crime_lookup["LSOA21CD"].is_unique
crime_lookup = crime_lookup.loc[crime_lookup["MSOA21CD"].isin(coverage_codes)]
crime_selected = crime_raw.merge(
    crime_lookup[["LSOA21CD", "MSOA21CD", "LAD22NM"]],
    left_on="LSOA code", right_on="LSOA21CD", how="inner", validate="many_to_one",
)
assert set(crime_selected["MSOA21CD"]) == coverage_codes
print(f"Regional download: {len(crime_raw):,} rows; 12 original fields; {len(crime_members)} monthly force CSVs")
print(f"Selected geography: {len(crime_selected):,} rows; {crime_selected['MSOA21CD'].nunique():,} MSOAs")
print(f"Download rows without an LSOA code: {crime_raw['LSOA code'].isna().sum():,}; not assignable to an MSOA")

Regional download: 846,166 rows; 12 original fields; 84 monthly force CSVs
Selected geography: 539,609 rows; 660 MSOAs
Download rows without an LSOA code: 7,786; not assignable to an MSOA


### Inspect fields and crime categories

I show the original 12 fields and Birmingham examples, followed by category totals for the selected geography. I have added source-file and geographical lookup fields internally for traceability; they are not new crime measures.

The source includes **anti-social behaviour**, which I keep separate from recorded crime. The “violence and sexual offences” category is combined by the publisher; this download cannot separate those two components.

In [40]:
crime_original_fields = [field for field in crime_raw if field != "source_file"]
crime_field_notes = [
    "Published identifier; may be missing or repeated", "Incident month",
    "Reporting police force", "Published force attribution",
    "Anonymised longitude", "Anonymised latitude", "Anonymised location description",
    "2021 LSOA code — geographical join key", "Published LSOA name",
    "Recorded category, including ASB", "Latest published outcome", "Additional context, often empty",
]
display(pd.DataFrame({"Source field": crime_original_fields, "What it contains": crime_field_notes}))
birmingham_crime = crime_selected.loc[crime_selected["LAD22NM"].eq("Birmingham")]
with pd.option_context("display.max_columns", None):
    display(birmingham_crime[crime_original_fields].head(5).reset_index(drop=True))
crime_category_counts = crime_selected["Crime type"].value_counts().rename_axis("category").reset_index(name="published_rows")
display(crime_category_counts)

,Source field,What it contains
0,Crime ID,Published identifier; may be missing or repeated
1,Month,Incident month
2,Reported by,Reporting police force
3,Falls within,Published force attribution
4,Longitude,Anonymised longitude
5,Latitude,Anonymised latitude
6,Location,Anonymised location description
7,LSOA code,2021 LSOA code — geographical join key
8,LSOA name,Published LSOA name
9,Crime type,"Recorded category, including ASB"


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2024-01,British Transport Police,British Transport Police,-1.828638,52.578809,On or near Four Oaks (Station),E01009427,Birmingham 004C,Vehicle crime,NaN,NaN
1,NaN,2024-01,British Transport Police,British Transport Police,-1.839273,52.528400,On or near Erdington (Station),E01009007,Birmingham 023D,Other theft,NaN,NaN
2,NaN,2024-01,British Transport Police,British Transport Police,-1.901709,52.516396,On or near Perry Barr (Station),E01009242,Birmingham 024E,Public order,NaN,NaN
3,NaN,2024-01,British Transport Police,British Transport Police,-1.913330,52.489921,On or near Jewellery Quarter (Station),E01033559,Birmingham 049E,Public order,NaN,NaN
4,NaN,2024-01,British Transport Police,British Transport Police,-1.892120,52.478692,On or near Birmingham Moor Street (Station),E01033561,Birmingham 050E,Criminal damage and arson,NaN,NaN


,category,published_rows
0,Violence and sexual offences,209057
1,Anti-social behaviour,55605
2,Shoplifting,45934
3,Vehicle crime,44158
4,Criminal damage and arson,40241
5,Other theft,35727
6,Public order,34963
7,Burglary,25330
8,Drugs,12513
9,Other crime,11543


### Check coverage and identifiers before counting

I check that each selected MSOA has records in all 12 months, and show the monthly totals by reporting force. This confirms records are present; it cannot prove that every incident was reported, recorded or supplied.

Some published crime IDs repeat, including records with different locations. I count **published rows** and retain a review table rather than arbitrarily keep the first location. Even identical rows can be ambiguous in this source, so I do not silently deduplicate them. These rates remain provisional pending review of the repeated IDs. Missing-location records stay in the raw download; I cannot assign them to areas or interpret them as zero crime.

I checked the historical changelog for reporting gaps. My downloaded files now contain records for every requested force in all 12 months, but that does not guarantee complete reporting.

In [41]:
crime_force_month = crime_raw.groupby(["Reported by", "Month"]).size().unstack(fill_value=0)
assert crime_force_month.shape == (7, 12)
assert crime_force_month.gt(0).all().all()
display(crime_force_month)
crime_months = crime_selected.groupby("MSOA21CD")["Month"].nunique()
assert crime_months.eq(12).all()
identified_crime = crime_selected.loc[crime_selected["Crime ID"].notna()]
crime_repeated_id_review = identified_crime.loc[
    identified_crime.duplicated(["Month", "Crime ID"], keep=False)
].copy()
crime_repeated_counts = crime_repeated_id_review.groupby("MSOA21CD").size()
print(f"MSOAs with records in all 12 months: {crime_months.eq(12).sum():,}")
print(f"Selected rows with a repeated month/ID: {len(crime_repeated_id_review):,}")
print(f"MSOAs containing these review rows: {crime_repeated_id_review['MSOA21CD'].nunique():,}")
print(f"Exact repeated original-field rows with non-missing ID: {identified_crime.duplicated(crime_original_fields).sum():,}")

Month,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
Reported by,,,,,,,,,,,,
British Transport Police,3376,3319,3481,3301,3519,3425,3526,3599,3311,3629,3545,3309
Derbyshire Constabulary,8216,8902,8910,8809,9491,9646,9543,9223,8724,9066,6964,7742
Leicestershire Police,8324,8243,8906,9128,9394,9490,9563,9357,9155,9172,8953,8216
Staffordshire Police,8504,7583,7888,9010,8857,7976,9243,8596,8502,8518,7696,7695
Warwickshire Police,3882,3780,3966,4209,4447,4221,4586,4217,4092,4260,3716,3684
West Mercia Police,7700,7654,8529,8540,9011,8917,8382,9063,8207,8500,8135,7757
West Midlands Police,27500,26067,28383,28495,31037,29865,31636,29180,28635,29074,26571,25723


MSOAs with records in all 12 months: 660
Selected rows with a repeated month/ID: 1,224
MSOAs containing these review rows: 65


Exact repeated original-field rows with non-missing ID: 0


### Keep two useful rates

I keep **published recorded-crime rows per 1,000 residents**, excluding anti-social behaviour, and **published ASB rows per 1,000 residents** separately. This avoids comparing raw totals from areas with different population sizes. These are incident-based rates, not percentages of residents who experienced crime.

I use the **Census 2021 usual-resident population** from TS007A, already supplied in the project, for both denominators. It is a fixed comparison base rather than an estimate of the 2024 population. Population change between those years can affect the rates.

My compact table has **area code, familiar local name and two rates**. The category breakdown, population denominator, monthly coverage and repeated-ID review counts remain in supporting files. I can explore particular categories later without adding them all to the feature table now.

A high rate in a shopping centre, city centre or transport hub may reflect visitors as well as residents. This is a measure of recorded activity relative to resident population, not a direct estimate of residents’ personal risk or the actual amount of all crime.

In [42]:
with ZipFile(raw_file("census2021-ts007a.zip")) as archive:
    with archive.open("census2021-ts007a-msoa.csv") as source_file:
        crime_population_raw = pd.read_csv(source_file)
assert crime_population_raw["geography code"].is_unique
assert crime_population_raw["date"].astype(str).eq("2021").all()
crime_population = crime_population_raw.set_index("geography code")["Age: Total"]
crime_area_counts = pd.crosstab(crime_selected["MSOA21CD"], crime_selected["Crime type"]).reindex(sorted(coverage_codes), fill_value=0)
crime_support = crime_area_counts.copy()
crime_support.index.name = "area_code"
crime_support = crime_support.reset_index()
crime_support.insert(1, "local_area_name", crime_support["area_code"].map(local_name_by_code))
crime_support["population_2021"] = crime_support["area_code"].map(crime_population)
assert crime_support["population_2021"].notna().all() and crime_support["population_2021"].gt(0).all()
crime_categories = [category for category in crime_area_counts.columns if category != "Anti-social behaviour"]
crime_support["recorded_crime_rows"] = crime_support[crime_categories].sum(axis=1)
crime_support["months_with_records"] = crime_support["area_code"].map(crime_months)
crime_support["repeated_id_review_rows"] = crime_support["area_code"].map(crime_repeated_counts).fillna(0).astype(int)
crime_display = crime_support[["area_code", "local_area_name"]].copy()
crime_display["recorded_crime_rows_per_1000_residents"] = 1000 * crime_support["recorded_crime_rows"] / crime_support["population_2021"]
crime_display["asb_rows_per_1000_residents"] = 1000 * crime_support["Anti-social behaviour"] / crime_support["population_2021"]
assert crime_display["area_code"].is_unique and crime_display["local_area_name"].notna().all()
assert crime_display.iloc[:, 2:].notna().all().all() and crime_display.iloc[:, 2:].ge(0).all().all()
assert crime_support[crime_area_counts.columns].sum().sum() == len(crime_selected)
print(f"Compact table: {len(crime_display):,} MSOAs; {len(crime_display.columns)} fields")
birmingham_crime_codes = set(birmingham_crime["MSOA21CD"])
display(crime_display.loc[crime_display["area_code"].isin(birmingham_crime_codes)].head(10).round(2))
print("Preview rounded only; exported rates retain precision. Repeated-ID caveat remains unresolved.")

Compact table: 660 MSOAs; 4 fields


Crime type,area_code,local_area_name,recorded_crime_rows_per_1000_residents,asb_rows_per_1000_residents
0,E02001827,Hill Hook,34.41,3.41
1,E02001828,Four Oaks,78.85,3.64
2,E02001829,Little Sutton,31.77,1.86
3,E02001830,Sutton Coldfield North,49.15,4.85
4,E02001831,Reddicap Heath,78.05,8.98
5,E02001832,Rectory Park,90.91,6.43
6,E02001833,Kingstanding North,61.32,6.40
7,E02001834,Old Oscott,80.48,4.91
8,E02001835,Sutton Coldfield South,118.39,18.06
9,E02001836,New Oscott,99.60,9.01


Preview rounded only; exported rates retain precision. Repeated-ID caveat remains unresolved.


### Check and save my crime dataset

I save the four-column provisional feature table in **`data/cleaned/Crime/`**, with category counts and denominators in the MSOA support file, force-month coverage in another file, and repeated-ID rows in a review file. The raw ZIP remains intact.

I retain timestamped backups on reruns and read every saved CSV back to verify it. This completes preparation of the three additional open sources: income, ethnicity and crime. Before using crime in modelling, I need to assess the identifier issue and the effects of missing locations, reporting practices and the later observation year. I have not joined these data to HLE or tested associations yet.

In [43]:
crime_folder = project_data("Cleaned/Crime")
crime_folder.mkdir(parents=True, exist_ok=True)
crime_coverage_export = crime_force_month.reset_index()
for filename, table in [
    ("crime_2024_msoa_provisional.csv", crime_display),
    ("crime_2024_msoa_support.csv", crime_support),
    ("crime_2024_force_month_coverage.csv", crime_coverage_export),
    ("crime_2024_repeated_id_review.csv", crime_repeated_id_review),
]:
    destination = crime_folder / filename
    if destination.exists():
        backup_folder = project_folder / "VS_CODE/Backups/Outputs/Crime"
        backup_folder.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        shutil.copy2(destination, backup_folder / f"{destination.stem} - {stamp}.csv")
    table.to_csv(destination, index=False)
    # Crime identifier fields are text; preserve them explicitly when reading back.
    reloaded = pd.read_csv(destination, dtype=table.dtypes.to_dict())
    pd.testing.assert_frame_equal(reloaded, table.reset_index(drop=True), check_names=False)
    print(f"Saved and verified {filename}: {len(reloaded):,} rows; {len(reloaded.columns)} fields")

Saved and verified crime_2024_msoa_provisional.csv: 660 rows; 4 fields
Saved and verified crime_2024_msoa_support.csv: 660 rows; 20 fields
Saved and verified crime_2024_force_month_coverage.csv: 7 rows; 13 fields
Saved and verified crime_2024_repeated_id_review.csv: 1,224 rows; 16 fields
